<!--HEADER-->
# EW-RIE*: Làm sạch ma trận tương quan có trọng số mũ bằng lý thuyết ma trận ngẫu nhiên
### Ứng dụng: danh mục phương sai nhỏ nhất trên HOSE — **phiên bản v3**
(điền sau khi chạy)

## Thay đổi trong v3 (so với v2.2)
1. **T1 hoàn thiện** — dữ liệu `data_hose/` (vnstock 5.2.2, nguồn VCI): giá đóng cửa điều chỉnh + khối lượng khớp lệnh 2015-01-05 → 2026-11-10, **493 mã gồm cả mã đã hủy niêm yết/chuyển sàn**; giá trị GD = giá × khối lượng. Dùng `security_master.csv` (lịch sử niêm yết HOSE) để chỉ giữ quan sát trong khoảng niêm yết (§2). Thêm **T1a** lịch sử niêm yết/hủy niêm yết, **T1b** kiểm tra chất lượng + trực quan hóa, **T1c** đo survivorship bias (§6).
2. **Lọc mã đứng giá là cấu hình chuẩn:** `ZERO_MAX = 0,20` — loại khỏi universe mã có > 20% phiên r = 0 trong cửa sổ quá khứ (chỉ dùng thông tin quá khứ). Toàn bộ §4–§6 chạy lại với cấu hình này; T2b so với không lọc và lọc 10%.
3. **Sửa EW-NL-LW khi N > T_eff** (`nl_lw_S`): v2 dùng công thức p ≤ n cho cả khi c = N/T_eff > 1 ⇒ co sai. v3 dùng nhánh p > n của Ledoit–Wolf (2020). Với c ≤ 1 kết quả **trùng hệt v2** (assert). Kiểm thử ở §1b (Test 2, Test 4) và T6b.
4. **Đối chứng mới EW-WeSpeR** (Oriol 2024, arXiv:2410.14413): khôi phục phổ quần thể của hiệp phương sai có trọng số (kiểu QuEST) rồi co phi tuyến bằng công thức oracle. Cài lại bằng numpy (§1, ô WeSpeR), kiểm thử ở §1b (Test 2, Test 3), đưa vào §4, T3, T6.
5. Mỗi hình kết quả có **nhận xét markdown ngay bên dưới** (viết sau khi chạy, số liệu lấy từ bảng/hình ngay trên). Giai đoạn thị trường cập nhật theo mẫu 2016–2026.
6. Bỏ các bộ nạp `github`/`vnstock` của v2 (dữ liệu đã cào sẵn ở `data_hose/`).

## Phân công nhóm

| Mã | Việc | Đầu ra | Mục trong notebook |
|---|---|---|---|
| **T1** ✔ v3 | Dữ liệu HOSE 2015–11/2026 (vnstock/VCI) + **lịch sử niêm yết, mã đã hủy niêm yết** để loại survivorship bias | `data_hose/` | §2 (T1a, T1b), §6 (T1c) |
| **T2** | Độ vững: chi phí GD thực tế (phí + thuế 0,1%), tần suất tái cân bằng 5/21/63 phiên, xử lý phiên không giao dịch (~18%) | Bảng độ vững | §6, ô T2 |
| **T3** | Đối chứng & kiểm định chuẩn: danh mục 1/N, mô hình nhân tố, DCC-GARCH; kiểm định chênh lệch phương sai Ledoit–Wolf (2011, HAC), Sharpe (Jobson–Korkie/Memmel) | Bảng kiểm định | §6, ô T3 |
| **T4** | Lý thuyết: chứng minh tính đúng của việc chọn nhánh, điều kiện ngoại lai, hiệu chỉnh N hữu hạn; **rà soát tính mới** (đọc toàn văn BBP 2016 §III và Phys. Rep. 2017) | Mục Methodology (LaTeX) | — |
| **T5** | Vì sao VS giúp trên HOSE nhưng không trong GARCH? (biến động riêng lẻ, tương quan đổi theo chế độ…) | Ghi chú + thí nghiệm | §6, ô T5 |
| **T6** | Mở rộng: N = 50/150, cửa sổ T, VN30/HNX | Bảng phụ lục | §5 (đổi CONFIG) |
| **T7** | Viết bài, nhắm tạp chí (kiểm tra hạng Q trên Scimago) | Bản thảo | — |

**Quy ước:** mỗi người làm trên bản sao notebook (`File → Save a copy in Drive`), ghi kết quả vào thư mục chung. Đổi tham số ở ô CONFIG, không sửa thư viện ở §1 trừ khi đã thống nhất.

## 0. Cài đặt & cấu hình

In [ ]:
import os, sys, subprocess, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import hashlib, json, inspect, platform, importlib.util
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB, "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# ======================= CONFIG =======================
CONFIG = dict(
    N=100,            # số mã thanh khoản nhất mỗi kỳ
    WINDOW=250,       # cửa sổ ước lượng (phiên)
    REB=21,           # tái cân bằng mỗi REB phiên
    COST=0.0025,      # chi phí 1 chiều (T2c: phí thực tế mua 0,15%/bán 0,25% ≡ 0,20%)
    HL_EW=60,         # half-life trọng số mũ
    HL_VOL=20,        # half-life chuẩn hóa biến động (VS)
    MAX_ABS_RET=0.15, # |r| > ngưỡng coi là lỗi dữ liệu (biên độ HOSE ±7%, phiên đầu/sau đình chỉ ±20%)
    ZERO_MAX=0.20,    # v3: loại mã đứng giá – tỷ lệ phiên r = 0 trong cửa sổ quá khứ > ZERO_MAX (1.0 = không lọc)
    QUICK=False,      # True: chạy nhanh với ít bộ ước lượng
)
# Dữ liệu HOSE 2015–2026 (repo data_HOSE, thư mục data_hose/): prices.csv, volumes.csv, security_master.csv, missing_tickers.csv
DATA_DIR = next((d for d in ("../data_hose", "./data_hose", ".") if os.path.exists(f"{d}/prices.csv") and os.path.exists(f"{d}/volumes.csv")),
                "https://raw.githubusercontent.com/chinguyenkhac/data_HOSE/main/data_hose")
WORKDIR = os.environ.get("RMT_WORKDIR", "/content/rmt" if IN_COLAB else "./rmt_work")
os.makedirs(WORKDIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR); CONFIG

## 1. Thư viện: các bộ ước lượng
v3: `nl_lw_S` có nhánh N > T_eff; thêm EW-WeSpeR (ô riêng). Mỗi bộ ước lượng nhận ma trận lợi suất `X` (T×N) và trả về ma trận hiệp phương sai N×N.

In [ ]:
# ---------- tiện ích ----------
def cov_to_corr(S):
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def renorm(C):
    d = np.sqrt(np.diag(C)); return C / np.outer(d, d)

# ---------- đối chứng cổ điển ----------
def est_sample(X): return np.cov(X, rowvar=False)

def est_lw(X): return LedoitWolf().fit(X).covariance_

# ---------- v3: NL-LW cho cả trường hợp p > n (N > T_eff) ----------
def nl_lw_S(S, n):
    # Ledoit & Wolf (2020), co phi tuyến giải tích; S: hiệp phương sai, n: cỡ mẫu (hiệu dụng, có thể không nguyên).
    # p <= n: công thức gốc (giống hệt v2).  p > n: nhánh "p > n" trong mã gốc analytical_shrinkage.m của LW (2020):
    # chỉ ⌊n⌋ trị riêng lớn nhất được xem là "khác 0" (dùng cho mật độ nhân), các trị riêng còn lại nhận chung d̃₀
    # (d̃₀ dựa trên Hilbert transform tại 0). v2 dùng công thức p <= n với c = p/n > 1 ⇒ mẫu số (1 − c − πcλH)² sai, co sai các trị riêng nhỏ.
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    m = p if p <= n else max(int(np.floor(n)), 2)
    lz = lam[p - m:]; h = n ** (-1 / 3)
    L = np.tile(lz, (m, 1)).T; H = h * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    if p <= n:
        d = lz / ((np.pi*c*lz*ft)**2 + (1 - c - np.pi*c*lz*Ht)**2)
    else:
        Hf0 = (1/np.pi) * (3/10/h**2 + 3/4/np.sqrt(5)/h * (1 - 1/5/h**2) * np.log((1 + np.sqrt(5)*h) / (1 - np.sqrt(5)*h))) * np.mean(1 / lz)
        d0 = 1 / (np.pi * (p - n) / n * Hf0)
        d1 = lz / (np.pi**2 * lz**2 * (ft**2 + Ht**2))
        d = np.concatenate([np.full(p - m, d0), d1])
    return (U * d) @ U.T

def nl_lw_S_v2(S, n):
    # BẢN v2 (chỉ để đối chiếu ở Test 2/4 và T6b): luôn dùng công thức p <= n, kể cả khi c = p/n > 1
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    L = np.tile(lam, (p, 1)).T; H = n ** (-1 / 3) * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    d = lam / ((np.pi*c*lam*ft)**2 + (1 - c - np.pi*c*lam*Ht)**2)
    return (U * d) @ U.T

def est_nllw(X):
    Xc = X - X.mean(0); n = len(X) - 1
    return nl_lw_S(Xc.T @ Xc / n, n)

def mp_corr(C, q):
    lam, V = np.linalg.eigh(C); N = len(C)
    lp = (1 - lam[-1] / N) * (1 + np.sqrt(q))**2
    noise = lam < lp; lam = lam.copy()
    if noise.any(): lam[noise] = lam[noise].mean()
    return renorm((V * lam) @ V.T)

def rie_corr(C, q):
    # RIE Bun–Bouchaud–Potters (2016) + hiệu chỉnh N hữu hạn (2017, §8). Đây là bản GỐC (đối chứng).
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10)
    z = lam - 1j / np.sqrt(N); g = (1 / (z[:, None] - lam[None, :])).mean(1)
    xi = lam / np.abs(1 - q + q*z*g)**2
    s2 = lam[0] / (1 - np.sqrt(q))**2 if q < 1 else lam.mean()
    lp, lm = s2 * (1 + np.sqrt(q))**2, s2 * (1 - np.sqrt(q))**2
    gmp = (z + s2*(q - 1) - np.sqrt(z - lm) * np.sqrt(z - lp)) / (2*q*z*s2)
    xi = xi * np.maximum(1, s2 * np.abs(1 - q + q*z*gmp)**2 / lam)
    xi *= N / xi.sum(); return renorm((V * xi) @ V.T)

def _corr_wrap(cleaner):
    def est(X):
        T, N = X.shape; C, sd = cov_to_corr(np.cov(X, rowvar=False))
        return cleaner(C, N / T) * np.outer(sd, sd)
    return est
est_mp, est_rie = _corr_wrap(mp_corr), _corr_wrap(rie_corr)

In [ ]:
# ---------- EW (trọng số mũ) và VS (chuẩn hóa biến động) ----------
def ew_weights(T, hl):
    w = 0.5 ** ((T - 1 - np.arange(T)) / hl); return w / w.sum()

def ew_corr(X, w):
    Xc = X - w @ X; S = (Xc * w[:, None]).T @ Xc
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def make_ew(cleaner, hl):
    # cleaner(C, w) -> tương quan đã làm sạch
    def est(X):
        w = ew_weights(len(X), hl); C, sd = ew_corr(X, w)
        return cleaner(C, w) * np.outer(sd, sd)
    return est

def ewma_standardize(X, hl):
    lam = 0.5 ** (1 / hl); s2 = X[:hl].var(0) + 1e-10; Z = np.empty_like(X)
    for t in range(len(X)):
        Z[t] = X[t] / np.sqrt(s2); s2 = lam * s2 + (1 - lam) * X[t]**2
    return Z, np.sqrt(s2)

def make_vs(corr_cleaner, hl_vol):
    # corr_cleaner(Z) -> ma trận tương quan đã làm sạch của lợi suất chuẩn hóa Z
    def est(X):
        Z, sig = ewma_standardize(X, hl_vol); Z = Z[hl_vol:]
        return corr_cleaner(Z) * np.outer(sig, sig)
    return est

In [ ]:
# ---------- EW-RIE*: RIE chính xác cho trọng số mũ (phương pháp đề xuất) ----------
_STATS = {"fail": 0, "total": 0}; _NULL_CACHE = {}

def _F(g, t, a, w):
    d = 1 - a[None, :] * g[:, None]; R = (w[None, :] / d).sum(1)
    return g * R - t, R, (w[None, :] * a[None, :] / d**2).sum(1)

def _solve_gamma(t, a, w, g0, iters=30):
    # Newton có tìm kiếm lùi cho  γ·R_W(γ) = t  (vector phức), a = N·w
    g = g0.copy(); f, R, dR = _F(g, t, a, w)
    for _ in range(iters):
        step = f / (R + g * dR); lam_ = np.ones(len(g))
        for _ in range(30):
            gn = g - lam_ * step; fn, Rn, dRn = _F(gn, t, a, w)
            bad = ~(np.abs(fn) < np.abs(f)) & (np.abs(f) > 1e-14)
            if not bad.any(): break
            lam_[bad] *= 0.5
        keep = bad | ~np.isfinite(fn)
        gn[keep], fn[keep], Rn[keep], dRn[keep] = g[keep], f[keep], R[keep], dR[keep]
        g, f, R, dR = gn, fn, Rn, dRn
        if np.max(np.abs(f)) < 1e-13: break
    return g, np.abs(f)

def _continuation(t, a, w, steps=60, strict=True):
    # Nhánh chính: γ ≈ t khi t→0, đi theo tia s·t (s: 0→1)
    g = 1e-3 * t
    for s in np.geomspace(1e-3, 1.0, steps):
        g, res = _solve_gamma(s * t, a, w, g)
    ok = np.isfinite(g) & (res < 1e-8)
    if strict and not ok.all(): raise RuntimeError("Newton không hội tụ")
    return g if strict else (g, ok)

def _xi(spec, x, w, N, eta):
    z = x - 1j * eta; g = (1 / (z[:, None] - spec[None, :])).mean(1); t = z * g - 1
    gam, ok = _continuation(t, N * w, w, strict=False)
    gam[~ok] = (t / (1 + N * (w**2).sum() * t))[~ok]       # dự phòng (thực tế: 0 lần trên HOSE)
    _STATS["fail"] += int((~ok).sum()); _STATS["total"] += len(ok)
    return x * gam.imag / t.imag

def _xi_outliers(lam, idx, w, N):
    a = N * w; out = np.empty(len(idx))
    for k, i in enumerate(idx):
        g = np.sum(1 / (lam[i] - np.delete(lam, i))) / N     # Stieltjes bỏ-một trên trục thực
        t = np.array([lam[i] * g - 1 + 0j])
        try:
            gam = _continuation(t, a, w); _, R, dR = _F(gam, t, a, w)
            out[k] = lam[i] * np.real(1 / (R + gam * dR))[0]   # ξ = λ·γ'(t)
        except RuntimeError:
            out[k] = _xi(lam, lam[i:i+1], w, N, 1 / np.sqrt(N))[0]
    return out

def _null_spectrum(N, w, reps=8, seed=0):
    key = (N, len(w), round(float(w[-1]), 12))
    if key not in _NULL_CACHE:
        rng = np.random.default_rng(seed)
        ev = [np.linalg.eigvalsh((Y * w[:, None]).T @ Y) for Y in (rng.standard_normal((len(w), N)) for _ in range(reps))]
        _NULL_CACHE[key] = np.sort(np.concatenate(ev))
    return _NULL_CACHE[key]

def ew_rie_corr(C, w, debias=True):
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10); eta = 1 / np.sqrt(N)
    null = _null_spectrum(N, w); s2 = np.median(lam) / np.median(null)
    D = lam[:, None] - lam[None, :]
    dens = (eta / (D**2 + eta**2)).sum(1) - 1 / eta            # mật độ lân cận (bỏ tự thân)
    out = np.where(dens < 0.1 / eta)[0]; bulk = np.setdiff1d(np.arange(N), out)
    xi = np.empty(N)
    xi[bulk] = _xi(lam, lam[bulk], w, N, eta)
    if len(out): xi[out] = _xi_outliers(lam, out, w, N)
    if debias and len(bulk):
        xi_null = _xi(null, lam[bulk] / s2, w, N, eta)
        xi[bulk] *= np.maximum(1, 1 / np.maximum(xi_null, 1e-6))
    xi = np.maximum(xi, 1e-8); xi *= N / xi.sum()
    return renorm((V * xi) @ V.T)

In [ ]:
# ---------- v3: EW-WeSpeR – đối chứng kiểu QuEST cho hiệp phương sai có trọng số (Oriol 2024, arXiv:2410.14413) ----------
# Mô hình: E = Σ_t w_t x_t x_tᵀ, x_t = C^{1/2} y_t, Σw = 1. Với r(e) = Σ_t w_t / (1 + N w_t e) (= R_W(−e)):
#   e(z) = (1/N) Σ_i τ_i / (τ_i r(e(z)) − z),      m(z) = (1/N) Σ_i 1 / (τ_i r(e(z)) − z)     (Im z > 0)
# m là Stieltjes của phổ mẫu tiệm cận (mật độ ρ = Im m / π); e = (1/N) tr C·G(z). Co phi tuyến "oracle":
#   d(λ) = lim u_λᵀ C u_λ = Im e(λ + i0) / Im m(λ + i0)      (trọng số đều ⇒ λ / |1 − q − qλ m|², Ledoit–Péché).
# WeSpeR = (1) tìm phổ quần thể τ sao cho phân vị của ρ_τ khớp trị riêng mẫu (như QuEST, nhưng với trọng số),
#          (2) áp d(λ_i) với τ đã khớp. Khác EW-RIE*: EW-RIE* dùng trực tiếp Stieltjes thực nghiệm (làm trơn η = N^{-1/2}),
#          WeSpeR dùng Stieltjes của MÔ HÌNH đã khớp ⇒ không cần η, không cần xử lý ngoại lai/hiệu chỉnh N hữu hạn riêng.
# Ghi chú: cài lại bằng numpy từ các phương trình trên (bản gốc của tác giả viết bằng PyTorch + autodiff; phiên làm việc này
# không truy cập được arXiv). τ tham số hóa: log τ tuyến tính từng khúc theo hạng (K nút) + các ngoại lai tự do;
# tối ưu bằng least_squares (Levenberg–Marquardt kiểu trf), sai số tương đối log λ̂_i − log λ_i; khởi tạo ấm từ kỳ trước.
_WSP = {"nfev": [], "cost": [], "warm": {}}

def _wsp_bins(w, nb=40):
    # nén phân phối trọng số thành nb nhóm (r chỉ phụ thuộc phân phối của w): giá trị TB và khối lượng Σw mỗi nhóm
    ws = np.sort(w); g = np.array_split(ws, min(nb, len(ws)))
    return np.array([x.mean() for x in g]), np.array([x.sum() for x in g])

def _wsp_solve(z, tau, wv, wm, N, e0=None, iters=40):
    # Newton (có tìm kiếm lùi, giữ Im e > 0) cho F(e) = e − mean τ/(τ r(e) − z), vector theo z
    def F(e):
        den = 1 + N * wv[None, :] * e[:, None]
        r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
        q = tau[None, :] * r[:, None] - z[:, None]
        f = e - (tau[None, :] / q).mean(1)
        df = 1 + (tau[None, :]**2 * dr[:, None] / q**2).mean(1)
        return f, df, q
    e = (tau[None, :] / (tau[None, :] - z[:, None])).mean(1) if e0 is None else e0.copy()
    f, df, q = F(e)
    for _ in range(iters):
        st = f / df; a = np.ones(len(e))
        for _ in range(25):
            en = e - a * st; fn, dfn, qn = F(en)
            bad = (~np.isfinite(fn)) | (np.abs(fn) > np.abs(f)) | (en.imag <= 0)
            bad &= np.abs(f) > 1e-13
            if not bad.any(): break
            a[bad] *= 0.5
        keep = bad
        en[keep], fn[keep], dfn[keep], qn[keep] = e[keep], f[keep], df[keep], q[keep]
        e, f, df, q = en, fn, dfn, qn
        if np.max(np.abs(f)) < 1e-11: break
    return e, (1 / q).mean(1), np.abs(f)

def _wsp_cold(x, eta, tau, wv, wm, N, steps=14):
    # tiếp tục theo η: từ η lớn (e ≈ mean τ/(τ − z)) xuống η đích ⇒ đúng nhánh Stieltjes (Im e > 0)
    e = None
    for s in np.geomspace(10 * max(x.max(), tau.max()), eta.min(), steps):
        e, m, res = _wsp_solve(x + 1j * np.maximum(eta, s), tau, wv, wm, N, e)
    return _wsp_solve(x + 1j * eta, tau, wv, wm, N, e)

def _wsp_quantiles(x, m, n_nz, N):
    # phân vị của phần khối lượng dương của ρ = Im m/π trên lưới x (hình thang), khớp với n_nz trị riêng khác 0
    rho = np.maximum(m.imag, 0) / np.pi; cdf = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * np.diff(x))])
    cdf /= cdf[-1]; u = (np.arange(n_nz) + 0.5) / n_nz
    return np.interp(u, cdf, x)

def _wsp_fwd(x, eta, tau, wv, wm, N, e0=None):
    # giải e trên lưới (ấm nếu có e0, nếu không/không hội tụ thì tiếp tục theo η) + các đạo hàm theo τ (đạo hàm ẩn)
    ok = False
    if e0 is not None:
        e, m, res = _wsp_solve(x + 1j * eta, tau, wv, wm, N, e0, iters=15); ok = np.isfinite(e).all() and res.max() < 1e-9
    if not ok: e, m, res = _wsp_cold(x, eta, tau, wv, wm, N)
    return e, m, res

def _wsp_jac_m(x, eta, e, tau, wv, wm, N):
    # dm/dτ_j (G×N):  F(e,τ) = e − mean τ/q, q = τ r(e) − z
    z = x + 1j * eta; den = 1 + N * wv[None, :] * e[:, None]
    r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
    q = tau[None, :] * r[:, None] - z[:, None]; q2 = q**2
    dF = 1 + (tau[None, :]**2 * dr[:, None] / q2).mean(1)
    de = -(z[:, None] / (N * q2)) / dF[:, None]
    A = -(tau[None, :] * dr[:, None] / q2).mean(1)
    return -(r[:, None] / (N * q2)) + A[:, None] * de

def ew_wesper_corr(C, w, K=8, max_nfev=25, G=350, return_info=False):
    from scipy.optimize import least_squares
    N = len(C); lam, V = np.linalg.eigh(C); T = len(w)
    n_nz = min(N, T - 1); lz = np.maximum(lam[N - n_nz:], 1e-10)          # trị riêng khác 0 (tăng dần)
    wv, wm = _wsp_bins(w)
    eta0 = 1 / np.sqrt(N); D = lz[:, None] - lz[None, :]                    # ngoại lai: như ew_rie_corr
    dens = (eta0 / (D**2 + eta0**2)).sum(1) - 1 / eta0
    n_out = int(min(np.sum(dens < 0.1 / eta0), 5)); nb = N - n_out
    knots = np.linspace(0, 1, K); rk = (np.arange(nb) + 0.5) / nb
    def tau_of(th):
        tb = np.exp(np.interp(rk, knots, np.cumsum(np.concatenate([[th[0]], np.exp(th[1:K])]))))
        return np.concatenate([tb, np.exp(th[K:])]) if n_out else tb
    x = np.unique(np.concatenate([np.geomspace(lz[0] * .25, lz[-1] * 1.6, G),
                                  *[lz[-k] * np.linspace(.6, 1.4, 60) for k in range(1, n_out + 1)]]))
    eta = 2e-3 * x; dx = np.diff(x); u = (np.arange(n_nz) + 0.5) / n_nz
    key = (N, T, K, n_out, round(float(w[-1]), 12))
    qb = np.maximum.accumulate(np.log(np.quantile(lz[:n_nz - n_out] if n_nz > n_out else lz, knots)))
    th_cold = np.concatenate([[qb[0]], np.log(np.maximum(np.diff(qb), 1e-3)), np.log(lz[n_nz - n_out:] * .9) if n_out else []])
    st = {}
    def evaluate(th):
        if st.get("th") is not None and np.array_equal(st["th"], th): return
        tau = tau_of(th); e, m, res = _wsp_fwd(x, eta, tau, wv, wm, N, st.get("e"))
        rho = np.maximum(m.imag, 0) / np.pi; raw = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * dx)])
        tot = raw[-1]; cdf = raw / tot; qh = np.interp(u, cdf, x)
        st.update(th=th.copy(), tau=tau, e=e, m=m, rho=rho, raw=raw, tot=tot, cdf=cdf, qh=qh, ok=np.isfinite(qh).all())
    def fun(th):
        evaluate(th); return np.log(st["qh"]) - np.log(lz) if st["ok"] else np.full(n_nz, 10.0)
    def jac(th):
        evaluate(th)
        dm = _wsp_jac_m(x, eta, st["e"], st["tau"], wv, wm, N); dr = dm.imag / np.pi
        draw = np.vstack([np.zeros(N), np.cumsum(0.5 * (dr[1:] + dr[:-1]) * dx[:, None], axis=0)])
        dcdf = (draw - st["cdf"][:, None] * draw[-1][None, :]) / st["tot"]
        k = np.clip(np.searchsorted(x, st["qh"]) - 1, 0, len(x) - 2); f = (st["qh"] - x[k]) / dx[k]
        dC = dcdf[k] * (1 - f)[:, None] + dcdf[k + 1] * f[:, None]
        pdf = np.maximum((st["rho"][k] * (1 - f) + st["rho"][k + 1] * f) / st["tot"], 1e-3 / (lz[-1]))
        Jt = -dC / pdf[:, None] / st["qh"][:, None]                              # ∂ log λ̂ / ∂τ
        h = 1e-6; t0 = tau_of(th)
        Tt = np.column_stack([(tau_of(th + h * np.eye(len(th))[j]) - t0) / h for j in range(len(th))])
        return Jt @ Tt
    fit = lambda th0: (st.clear(), least_squares(fun, th0, jac=jac, method="trf", max_nfev=max_nfev, xtol=1e-4, ftol=1e-4))[1]
    sol = fit(_WSP["warm"].get(key, th_cold))                 # khởi tạo ấm từ kỳ trước (cùng N, T, h) nếu có
    if key in _WSP["warm"] and np.sqrt(np.mean(sol.fun**2)) > 0.05:  # ấm hỏng (cực tiểu địa phương) ⇒ khớp lại từ đầu, giữ nghiệm tốt hơn
        s2 = fit(th_cold); sol = s2 if np.sum(s2.fun**2) < np.sum(sol.fun**2) else sol
    _WSP["warm"][key] = sol.x; _WSP["nfev"].append(sol.nfev); _WSP["cost"].append(np.sqrt(np.mean(sol.fun**2)))
    tau = tau_of(sol.x)
    # co: d(λ_i) = Im e / Im m tại λ_i + iη (η nhỏ; ở khe/ngoài giá đỡ tỷ số → e'(λ)/m'(λ), vẫn xác định)
    ee, mm, _ = _wsp_cold(lz, 1e-4 * lz, tau, wv, wm, N); d = ee.imag / mm.imag
    xi = np.empty(N); xi[N - n_nz:] = np.maximum(d, 1e-8)
    if n_nz < N:   # không gian null (N ≥ T): giữ vết (tr C = N) cho các hướng null
        xi[:N - n_nz] = max((N - xi[N - n_nz:].sum()) / (N - n_nz), 1e-3 * np.mean(xi[N - n_nz:]))
    xi *= N / xi.sum(); out = renorm((V * xi) @ V.T)
    return (out, dict(tau=tau, xi=xi, lam=lam, rmse=_WSP["cost"][-1], nfev=sol.nfev, n_out=n_out)) if return_info else out


# Bọc an toàn + cache theo (C, w): cùng cửa sổ ở hai chế độ chỉ ước lượng 1 lần; lỗi số ⇒ dự phòng EW-RIE* (được đếm)
_WSP_SAFE = {"fallback": 0, "total": 0}; _WSP_CACHE = {}
def ew_wesper_safe(C, w):
    key = (hashlib.md5(C.tobytes()).hexdigest(), len(w), round(float(w[-1]), 14))
    if key not in _WSP_CACHE:
        _WSP_SAFE["total"] += 1
        try:
            out = ew_wesper_corr(C, w); assert np.isfinite(out).all()
        except Exception:
            _WSP_SAFE["fallback"] += 1; out = ew_rie_corr(C, w)
        _WSP_CACHE[key] = out
    return _WSP_CACHE[key]

In [ ]:
# ---------- Danh sách bộ ước lượng ----------
def build_estimators(cfg):
    hl, hv = cfg["HL_EW"], cfg["HL_VOL"]
    E = {
        "Sample": est_sample,
        "Ledoit-Wolf": est_lw,
        "NL-LW": est_nllw,
        "MP-clip": est_mp,
        "RIE": est_rie,
        "VS-NL-LW": make_vs(lambda Z: renorm(nl_lw_S(np.cov(Z, rowvar=False), len(Z) - 1)), hv),
        "VS-RIE": make_vs(lambda Z: rie_corr(np.corrcoef(Z, rowvar=False), Z.shape[1] / len(Z)), hv),
        "EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl),        # v3: đúng cả khi N > T_eff
        "EW-RIE (q_eff)": make_ew(lambda C, w: rie_corr(C, len(C) * (w**2).sum()), hl),
        "EW-WeSpeR": make_ew(ew_wesper_safe, hl),                                          # v3: đối chứng Oriol (2024)
        "EW-RIE*": make_ew(ew_rie_corr, hl),
    }
    if cfg["QUICK"]:
        E = {k: E[k] for k in ["Sample", "NL-LW", "RIE", "EW-NL-LW", "EW-WeSpeR", "EW-RIE*"]}
    return E

## 1b. Kiểm thử (chạy trước mọi thay đổi)
1) Trọng số đều ⇒ EW-RIE\* trùng RIE cổ điển. 2) So với oracle trên dữ liệu tổng hợp đã biết C thật — v3 thêm h = 15 (N = 100 > T_eff ≈ 43), EW-NL-LW bản v2/v3 và EW-WeSpeR.
3) Mô hình thuận của WeSpeR: phân vị phổ mẫu dự báo từ phổ quần thể τ khớp trị riêng mô phỏng. 4) NL-LW nhánh p > n: đường co d(λ) so với oracle.

In [ ]:
rng = np.random.default_rng(0); N_, T_ = 100, 250
X = rng.standard_normal((T_, N_)); Cs = np.corrcoef(X, rowvar=False); lam = np.linalg.eigvalsh(Cs)
z = lam - 0.1j; g = (1 / (z[:, None] - lam[None, :])).mean(1); q = N_ / T_
err = np.max(np.abs(_xi(lam, lam, np.full(T_, 1 / T_), N_, 0.1) - lam / np.abs(1 - q + q*z*g)**2) / lam)
print(f"[Test 1] trọng số đều: sai lệch tương đối max = {err:.1e}"); assert err < 1e-10

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def factor_corr(N, rng):
    sec = rng.integers(0, 8, N); B = np.zeros((N, 9))
    B[:, 0] = rng.uniform(.4, .8, N); B[np.arange(N), 1 + sec] = rng.uniform(.2, .5, N)
    return renorm(B @ B.T + np.diag(rng.uniform(.3, .6, N)))

rows = []
for hl in (15, 30, 60):
    w = ew_weights(T_, hl); acc = {}
    for rep in range(6):
        Ct = factor_corr(N_, rng); X = rng.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T
        C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
        Co = renorm((V * np.einsum("ij,jk,ki->i", V.T, Ct, V)) @ V.T)    # oracle RIE
        wo = gmv(Ct); best = wo @ Ct @ wo
        for k, Ch in {"EW-Sample": C, "EW-RIE (q_eff)": rie_corr(C, N_ * (w**2).sum()),
                      "EW-NL-LW (v2)": renorm(nl_lw_S_v2(C, 1 / (w**2).sum())), "EW-NL-LW": renorm(nl_lw_S(C, 1 / (w**2).sum())),
                      "EW-WeSpeR": ew_wesper_corr(C, w), "EW-RIE*": ew_rie_corr(C, w), "Oracle": Co}.items():
            wh = gmv(Ch); acc.setdefault(k, []).append(wh @ Ct @ wh / best)
    rows += [dict(hl=hl, method=k, risk_ratio=np.mean(v)) for k, v in acc.items()]
t2 = pd.DataFrame(rows).pivot(index="method", columns="hl", values="risk_ratio").round(3)
teff = {hl: 1 / (ew_weights(T_, hl)**2).sum() for hl in (15, 30, 60)}
print("[Test 2] rủi ro GMV / rủi ro tối ưu (1 = hoàn hảo), mô hình nhân tố, N=100, T=250; T_eff =",
      {h: round(v) for h, v in teff.items()}); display(t2)
assert (t2.loc["EW-RIE*", [30, 60]] < t2.loc["EW-RIE (q_eff)", [30, 60]]).all()
assert (t2.loc["EW-NL-LW"] <= t2.loc["EW-NL-LW (v2)"] + 1e-9).all()                 # bản sửa không kém v2
assert np.isclose(t2.loc["EW-NL-LW", 60], t2.loc["EW-NL-LW (v2)", 60])              # N < T_eff: trùng v2
assert (t2.loc["EW-WeSpeR"] / t2.loc["Oracle"] < 1.03).all()                        # WeSpeR trong 3% của oracle

In [ ]:
# [Test 3] Mô hình thuận WeSpeR: cho τ (1 mode thị trường + khối), so phân vị mật độ tiệm cận với trị riêng mô phỏng (TB 20 lần)
rng3 = np.random.default_rng(1); fig, ax = plt.subplots(1, 3, figsize=(14, 3.8)); rows3 = []
tau3 = np.concatenate([np.geomspace(.3, 2, N_ - 1), [25.]]); tau3 *= N_ / tau3.sum()
for a, hl in zip(ax, (None, 60, 15)):
    w = np.full(T_, 1 / T_) if hl is None else ew_weights(T_, hl)
    ev = np.sort(np.mean([np.linalg.eigvalsh((Y * w[:, None]).T @ Y) for Y in (rng3.standard_normal((T_, N_)) * np.sqrt(tau3) for _ in range(20))], 0))
    x = np.unique(np.concatenate([np.geomspace(ev[0] * .25, ev[-1] * 1.6, 1500), ev[-1] * np.linspace(.6, 1.4, 80)])); wv, wm = _wsp_bins(w)
    e, m, res = _wsp_cold(x, 2e-3 * x, tau3, wv, wm, N_); qh = _wsp_quantiles(x, m, N_, N_)
    rows3.append(dict(hl="đều" if hl is None else hl, T_eff=1 / (w**2).sum(), max_residual=res.max(),
                      median_rel_err=np.median(np.abs(qh / ev - 1)), lam1_model=qh[-1], lam1_sim=ev[-1]))
    lab = "trọng số đều" if hl is None else f"EW h={hl}"; bulk = x < ev[-2] * 1.3
    a.hist(ev[:-1], bins=30, density=True, color="#c9c9c9", label="trị riêng mô phỏng (khối)")
    a.plot(x[bulk], m.imag[bulk] / np.pi * N_ / (N_ - 1), color="#8e44ad", lw=2, label="mật độ tiệm cận WeSpeR")
    a.set_title(f"{lab}: T_eff={1 / (w**2).sum():.0f}, λ₁ mô hình {qh[-1]:.1f} / mô phỏng {ev[-1]:.1f}", fontsize=9); a.legend(fontsize=7)
plt.tight_layout(); plt.show(); t3 = pd.DataFrame(rows3); display(t3.round(4))
assert (t3.max_residual < 1e-9).all() and (t3.median_rel_err < 0.05).all()

<!--CMT:test3-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# [Test 4] Đường co d(λ) khi N > T_eff (h = 15, T_eff ≈ 43): NL-LW v2 vs v3, EW-RIE*, EW-WeSpeR, oracle uᵢᵀ C uᵢ
rng4 = np.random.default_rng(4); w = ew_weights(T_, 15); Ct = factor_corr(N_, rng4)
X = rng4.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T; C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
orc = np.einsum("ij,jk,ki->i", V.T, Ct, V)
dd = {"EW-NL-LW (v2)": renorm(nl_lw_S_v2(C, 1 / (w**2).sum())), "EW-NL-LW (v3)": renorm(nl_lw_S(C, 1 / (w**2).sum())),
      "EW-RIE*": ew_rie_corr(C, w), "EW-WeSpeR": ew_wesper_corr(C, w)}
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter(lam, orc, s=12, color="#52514e", label="oracle uᵢᵀCuᵢ", zorder=3)
for (k, Ch), col in zip(dd.items(), ["#eda100", "#eb6834", "#1baf7a", "#8e44ad"]):
    ax[0].plot(lam, np.einsum("ij,jk,ki->i", V.T, Ch, V), lw=2, color=col, label=k)
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("trị riêng mẫu λᵢ (tương quan EW)"); ax[0].set_ylabel("trị riêng đã làm sạch")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3); ax[0].set_title(f"N={N_}, T={T_}, h=15 ⇒ T_eff={1 / (w**2).sum():.0f}, q_eff={N_ * (w**2).sum():.2f}", fontsize=10)
r4 = {k: (lambda wh: wh @ Ct @ wh / (gmv(Ct) @ Ct @ gmv(Ct)))(gmv(Ch)) for k, Ch in dd.items()}
ax[1].barh(list(r4), list(r4.values()), color=["#eda100", "#eb6834", "#1baf7a", "#8e44ad"]); ax[1].axvline(1, color="#52514e", lw=.8)
for i, v in enumerate(r4.values()): ax[1].text(v, i, f" {v:.2f}", va="center", fontsize=9)
ax[1].set_xscale("log"); ax[1].set_title("rủi ro GMV / rủi ro tối ưu (log)", fontsize=10); plt.tight_layout(); plt.show()
# Kiểm tra thêm: với c <= 1 hai bản trùng nhau tuyệt đối; với mẫu không trọng số N > T (est_nllw) bản v3 hữu hạn, PSD
Xs = rng4.standard_normal((250, 100)); S_ = np.cov(Xs, rowvar=False); assert np.allclose(nl_lw_S(S_, 249), nl_lw_S_v2(S_, 249))
Xw = rng4.standard_normal((80, 100)); ev_ = np.linalg.eigvalsh(est_nllw(Xw)); assert np.isfinite(ev_).all() and ev_[0] > 0
print("[Test 4] c <= 1: v3 ≡ v2 ✔ | N=100 > T=80: est_nllw hữu hạn, xác định dương ✔ | tỷ số rủi ro:", {k: round(v, 3) for k, v in r4.items()})

<!--CMT:test4-->
**Nhận xét.** (điền sau khi chạy)

## 2. Dữ liệu (T1)
Nguồn: repo `data_HOSE/data_hose` — cào bằng vnstock 5.2.2 (VCI), lịch sử niêm yết từ HSX + Vietstock + VCI (xem `README.md`, `QA.md`, `ENV.json` trong thư mục dữ liệu).
Kết quả chuẩn: `P` (giá đóng cửa điều chỉnh, đồng; dòng = ngày, cột = mã), `Q` (khối lượng khớp), `V = P × Q` (giá trị GD xấp xỉ), `MASTER` (mỗi dòng = một giai đoạn niêm yết HOSE).
Quy tắc T1: chỉ giữ quan sát nằm trong khoảng niêm yết HOSE theo `security_master`; ngày không giao dịch/đình chỉ = NaN (không nội suy); mã hủy niêm yết **được giữ** (không survivorship bias).

In [ ]:
def load_hose(d):
    P = pd.read_csv(f"{d}/prices.csv", index_col=0, parse_dates=True).sort_index()
    Q = pd.read_csv(f"{d}/volumes.csv", index_col=0, parse_dates=True).reindex(index=P.index, columns=P.columns)
    M = pd.read_csv(f"{d}/security_master.csv", parse_dates=["listing_date", "delisting_date"])
    miss = pd.read_csv(f"{d}/missing_tickers.csv")
    return P, Q, M, miss

P_RAW, Q, MASTER_RAW, MISSING = load_hose(DATA_DIR)
MASTER = MASTER_RAW.drop_duplicates(subset=["ticker", "listing_date", "delisting_date"]).reset_index(drop=True)
# mặt nạ khoảng niêm yết HOSE (một mã có thể có nhiều giai đoạn)
INWIN = pd.DataFrame(False, index=P_RAW.index, columns=P_RAW.columns)
for r in MASTER.itertuples():
    if r.ticker in INWIN:
        INWIN.loc[(INWIN.index >= r.listing_date) & ((INWIN.index <= r.delisting_date) if pd.notna(r.delisting_date) else True), r.ticker] = True
OUTSIDE = P_RAW.notna() & ~INWIN
P = P_RAW.where(INWIN); Q = Q.where(INWIN); V = P * Q
P = P[P.notna().sum(1) >= 20]; V = V.reindex(P.index); Q = Q.reindex(P.index)
DELISTED = set(MASTER.loc[MASTER.delisting_date.notna(), "ticker"]) - set(MASTER.loc[MASTER.delisting_date.isna(), "ticker"])
LAST_DELIST = MASTER.dropna(subset=["delisting_date"]).groupby("ticker").delisting_date.max()
print(f"{P.shape[1]} mã có giá, {P.shape[0]} phiên: {P.index[0].date()} → {P.index[-1].date()} | security_master: {len(MASTER_RAW)} dòng "
      f"({len(MASTER_RAW) - len(MASTER)} dòng trùng bị bỏ), {MASTER.ticker.nunique()} mã, {len(DELISTED)} mã đã rời HOSE | "
      f"thiếu giá: {len(MISSING)} mã | quan sát ngoài khoảng niêm yết (đã loại): {int(OUTSIDE.sum().sum())}")

### T1a – Lịch sử niêm yết và mã hủy niêm yết

In [ ]:
days = P.index; listed = pd.Series(0, index=days)
for r in MASTER.itertuples():
    listed[(days >= r.listing_date) & ((days <= r.delisting_date) if pd.notna(r.delisting_date) else True)] += 1
withpx = P.notna().sum(1)
RN = {"bat_buoc": "bắt buộc", "hop_nhat": "hợp nhất/sáp nhập", "tu_nguyen": "tự nguyện", "chuyen_san": "chuyển sàn", "chua_ro": "chưa rõ"}
dl = MASTER.dropna(subset=["delisting_date"]); dl = dl[dl.delisting_date >= P.index[0]]
DY = dl.assign(year=dl.delisting_date.dt.year, reason=dl.reason.map(RN).fillna("chưa rõ")).pivot_table(index="year", columns="reason", values="ticker", aggfunc="count", fill_value=0)
NL = MASTER[MASTER.listing_date >= P.index[0]].listing_date.dt.year.value_counts().sort_index()
pxed = set(P.columns)
T1A = pd.DataFrame({"Niêm yết mới": NL, "Hủy niêm yết": DY.sum(1),
                    "Hủy – có giá": dl[dl.ticker.isin(pxed)].delisting_date.dt.year.value_counts(),
                    "Hủy – thiếu giá": dl[~dl.ticker.isin(pxed)].delisting_date.dt.year.value_counts()}).fillna(0).astype(int)
T1A.loc["Tổng"] = T1A.sum(); display(T1A.T)
print("Lý do hủy niêm yết (2015–2026):", dl.reason.map(RN).fillna("chưa rõ").value_counts().to_dict(),
      "| sàn đến sau khi rời HOSE:", dl.exchange_after.value_counts().to_dict())
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(listed, color="#52514e", lw=2, label="đang niêm yết (security_master)")
ax[0].plot(withpx, color="#2a78d6", lw=1.2, label="có giá trong ngày (prices.csv)")
ax[0].fill_between(days, withpx, listed, color="#eb6834", alpha=.25, label="niêm yết nhưng không có giá (đình chỉ/thiếu)")
ax[0].set_title("Số mã trên HOSE theo ngày", fontsize=10); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
DY.plot.bar(stacked=True, ax=ax[1], color=["#eb6834", "#2a78d6", "#9aa0a6", "#1baf7a", "#eda100"][:DY.shape[1]], width=.75)
ax[1].plot(range(len(DY)), NL.reindex(DY.index).fillna(0).values, color="#52514e", marker="o", lw=1.5, label="niêm yết mới")
ax[1].set_title("Số mã rời HOSE theo năm và lý do (đường: niêm yết mới)", fontsize=10); ax[1].legend(fontsize=8); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

<!--CMT:t1a-->
**Nhận xét.** (điền sau khi chạy)

### T1b – Kiểm tra chất lượng và đánh giá tổng quan bộ dữ liệu

In [ ]:
R_all = np.log(P).diff(); r_s = np.expm1(R_all); nobs = int(R_all.notna().sum().sum())
inw = INWIN.reindex(P.index)
QA = pd.Series({
    "Số phiên": len(P), "Số mã có giá": P.shape[1], "Số mã trong security_master": MASTER.ticker.nunique(),
    "Mã thiếu giá (missing_tickers)": len(MISSING), "Mã thiếu giá đã hủy niêm yết": int(MISSING.delisting_date.notna().sum()),
    "Dòng trùng trong security_master": len(MASTER_RAW) - len(MASTER),
    "Quan sát ngoài khoảng niêm yết (đã loại)": int(OUTSIDE.sum().sum()),
    "% ô NaN trong khoảng niêm yết": 100 * (inw & P.isna()).sum().sum() / inw.sum().sum(),
    "Số quan sát lợi suất": nobs, "% phiên r = 0": 100 * (R_all == 0).sum().sum() / nobs,
    "% |r| > 7,5% (ngoài biên độ ±7%)": 100 * (r_s.abs() > .075).sum().sum() / nobs,
    "% |r| > 15% (coi là lỗi, đặt = 0)": 100 * (R_all.abs() > CONFIG["MAX_ABS_RET"]).sum().sum() / nobs,
    "% quan sát giá điều chỉnh < 2.000đ": 100 * (P < 2000).sum().sum() / P.notna().sum().sum(),
    "Phiên khối lượng = 0 mà có giá": int(((Q == 0) & P.notna()).sum().sum()),
    "Giá ≤ 0 hoặc NaN/inf trong giá": int((P <= 0).sum().sum() + np.isinf(P).sum().sum()),
})
display(QA.to_frame("giá trị").style.format(precision=3, decimal=",", thousands=" "))
print("Mã có quan sát ngoài khoảng niêm yết:", OUTSIDE.sum()[OUTSIDE.sum() > 0].sort_values(ascending=False).head(8).to_dict())
# Hình 1: độ phủ dữ liệu theo tháng (mã xếp theo ngày có giá đầu tiên) + tỷ lệ NaN theo năm
cov_m = P.notna().resample("ME").mean().T; first = P.apply(lambda s: s.first_valid_index()); cov_m = cov_m.loc[first.sort_values().index]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw=dict(width_ratios=[1.6, 1]))
im = ax[0].imshow(cov_m.values, aspect="auto", cmap="Blues", interpolation="nearest", vmin=0, vmax=1)
yr = cov_m.columns.year; tk = [i for i in range(len(yr)) if i == 0 or yr[i] != yr[i - 1]]
ax[0].set_xticks(tk, yr[tk], fontsize=8); ax[0].set_ylabel(f"{P.shape[1]} mã (xếp theo ngày có giá đầu tiên)"); ax[0].set_yticks([])
dlm = [i for i, t in enumerate(cov_m.index) if t in DELISTED]; ax[0].scatter(np.full(len(dlm), -1.5), dlm, s=2, color="#eb6834", clip_on=False)
plt.colorbar(im, ax=ax[0], label="tỷ lệ phiên có giá trong tháng"); ax[0].set_title("Độ phủ dữ liệu (chấm cam: mã đã rời HOSE)", fontsize=10)
nan_y = (inw & P.isna()).groupby(P.index.year).sum().sum(1) / inw.groupby(P.index.year).sum().sum(1) * 100
z_y = (R_all == 0).groupby(P.index.year).sum().sum(1) / R_all.notna().groupby(P.index.year).sum().sum(1) * 100
ax[1].bar(nan_y.index - .2, nan_y.values, .4, color="#eb6834", label="% NaN trong khoảng niêm yết")
ax[1].bar(z_y.index + .2, z_y.values, .4, color="#2a78d6", label="% phiên r = 0")
ax[1].legend(fontsize=8); ax[1].grid(axis="y", alpha=.3); ax[1].set_title("Phiên thiếu giá và phiên đứng giá theo năm", fontsize=10)
plt.tight_layout(); plt.show()
print("% NaN theo năm:", nan_y.round(1).to_dict()); print("% r=0 theo năm:", z_y.round(1).to_dict())

<!--CMT:t1b_cov-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# Hình 2: phiên đứng giá (r = 0) và thanh khoản – cơ sở cho lọc ZERO_MAX
zr_t = (R_all == 0).sum() / R_all.notna().sum(); mv = V.median()
liq_q = pd.qcut(mv.rank(method="first"), 5, labels=["Q1 (kém nhất)", "Q2", "Q3", "Q4", "Q5 (tốt nhất)"])
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(zr_t * 100, bins=40, color="#9aa0a6"); ax[0].axvline(100 * CONFIG["ZERO_MAX"], color="#eb6834", lw=2, label=f"ZERO_MAX = {CONFIG['ZERO_MAX']:.0%}")
ax[0].set_xlabel("% phiên r = 0 của mỗi mã (toàn mẫu)"); ax[0].set_ylabel("số mã"); ax[0].legend(fontsize=8); ax[0].set_title("Phân bố tỷ lệ phiên đứng giá theo mã", fontsize=10)
g = zr_t.groupby(liq_q).median() * 100; ax[1].bar(g.index.astype(str), g.values, color="#2a78d6")
for i, v in enumerate(g.values): ax[1].text(i, v, f"{v:.0f}%", ha="center", va="bottom", fontsize=8)
ax[1].set_title("Trung vị % phiên r = 0 theo nhóm thanh khoản (GTGD trung vị)", fontsize=10); ax[1].tick_params(axis="x", labelsize=8)
ax[2].scatter(mv / 1e9, zr_t * 100, s=8, alpha=.5, color="#52514e"); ax[2].set_xscale("log")
ax[2].axhline(100 * CONFIG["ZERO_MAX"], color="#eb6834", lw=1.5); ax[2].set_xlabel("GTGD trung vị (tỷ đồng/phiên, log)"); ax[2].set_ylabel("% phiên r = 0")
ax[2].set_title("Thanh khoản vs đứng giá", fontsize=10); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Mã có > {CONFIG['ZERO_MAX']:.0%} phiên r=0 (toàn mẫu): {(zr_t > CONFIG['ZERO_MAX']).sum()}/{len(zr_t)}; trung vị theo nhóm thanh khoản:", g.round(1).to_dict(),
      f"| tương quan hạng (GTGD, %r=0) = {mv.rank().corr(zr_t.rank()):.2f}")

<!--CMT:t1b_zero-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# Hình 3: phân phối lợi suất ngày, phiên vượt biên độ theo năm (tách mã giá < 2.000đ – nhiễu bước giá 100đ của VCI)
big = r_s.abs() > .075; low = P.shift(1) < 2000
by = pd.DataFrame({"giá ≥ 2.000đ": (big & ~low).groupby(P.index.year).sum().sum(1), "giá < 2.000đ": (big & low).groupby(P.index.year).sum().sum(1)})
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
rv = r_s.values.ravel(); rv = rv[np.isfinite(rv)]
ax[0].hist(np.clip(rv, -.3, .3), bins=241, color="#2a78d6", log=True)
for x_ in (-.07, .07): ax[0].axvline(x_, color="#eb6834", lw=1)
ax[0].set_title("Lợi suất ngày (cắt ±30%, trục y log); đường cam: biên độ ±7%", fontsize=10)
by.plot.bar(stacked=True, ax=ax[1], color=["#2a78d6", "#eda100"], width=.75); ax[1].set_title("Số phiên |r| > 7,5% theo năm", fontsize=10); ax[1].set_xlabel("")
k_ = (r_s.abs() > .075).sum().sort_values(ascending=False).head(12); ax[2].barh(k_.index[::-1], k_.values[::-1], color=["#eb6834" if t in DELISTED else "#9aa0a6" for t in k_.index[::-1]])
ax[2].set_title("12 mã nhiều phiên |r| > 7,5% (cam: đã rời HOSE)", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Phiên |r|>7,5%: {int(big.sum().sum())}, trong đó giá < 2.000đ: {int((big & low).sum().sum())}; |r|>15% (đặt 0): {int((R_all.abs() > CONFIG['MAX_ABS_RET']).sum().sum())}")
print(f"Phân vị lợi suất ngày: 0,1% = {np.quantile(rv, .001):.4f}, 99,9% = {np.quantile(rv, .999):.4f}, độ nhọn dư = {pd.Series(rv).kurt():.1f}")

<!--CMT:t1b_ret-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# Hình 4: bối cảnh thị trường + số mã đủ điều kiện mỗi kỳ tái cân bằng theo quy tắc chọn universe
R_q = np.log(P).diff(); R_q = np.expm1(R_q.mask(R_q.abs() > CONFIG["MAX_ABS_RET"], 0.0)); Wn = CONFIG["WINDOW"]; elig = []
for s in range(Wn + 1, len(P) - 1, CONFIG["REB"]):
    win = R_q.iloc[s - Wn:s]; ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna(); zs = (win == 0).mean()
    elig.append(dict(date=P.index[s], cov95=int(ok.sum()), z20=int((ok & (zs <= .2)).sum()), z10=int((ok & (zs <= .1)).sum())))
ELIG = pd.DataFrame(elig).set_index("date")
ew_idx = (1 + R_q.mean(1, skipna=True).fillna(0)).cumprod(); val_d = V.sum(1) / 1e12
PERIODS_PLOT = [("2016-17", "2016-01-01", "2017-12-31"), ("2018-19", "2018-01-01", "2019-12-31"), ("COVID", "2020-01-01", "2020-03-31"),
                ("2020-21", "2020-04-01", "2021-12-31"), ("2022", "2022-01-01", "2022-12-31"), ("2023-24", "2023-01-01", "2024-12-31"), ("2025-26", "2025-01-01", "2026-12-31")]
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(ew_idx, color="#52514e", lw=1.8, label="chỉ số đồng tỷ trọng mọi mã (lợi suất TB ngày)"); ax2 = ax[0].twinx()
ax2.fill_between(val_d.index, val_d.rolling(21).mean(), color="#2a78d6", alpha=.25, label="GTGD toàn sàn (nghìn tỷ đồng/phiên, TB 21 phiên)")
for i, (lab, a_, b_) in enumerate(PERIODS_PLOT):
    ax[0].axvspan(pd.Timestamp(a_), min(pd.Timestamp(b_), P.index[-1]), color=["#f3f3f3", "#ffffff"][i % 2], zorder=0)
    ax[0].text(pd.Timestamp(a_), ew_idx.max() * .97, lab, fontsize=7)
ax[0].set_title("Bối cảnh: chỉ số EW và thanh khoản HOSE", fontsize=10); ax[0].legend(fontsize=8, loc="center left"); ax2.legend(fontsize=8, loc="lower right")
ax[1].plot(ELIG.cov95, color="#9aa0a6", lw=2, label="phủ ≥ 95% phiên (v2)"); ax[1].plot(ELIG.z20, color="#1baf7a", lw=2, label="+ r=0 ≤ 20% (chuẩn v3)")
ax[1].plot(ELIG.z10, color="#eda100", lw=2, label="+ r=0 ≤ 10%"); ax[1].axhline(CONFIG["N"], color="#eb6834", ls="--", label=f"N = {CONFIG['N']}")
ax[1].set_title("Số mã đủ điều kiện mỗi kỳ tái cân bằng", fontsize=10); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print("Số mã đủ ĐK (min / trung vị / max):", {c: (int(ELIG[c].min()), int(ELIG[c].median()), int(ELIG[c].max())) for c in ELIG},
      f"| số kỳ < N: z20 = {(ELIG.z20 < CONFIG['N']).sum()}, z10 = {(ELIG.z10 < CONFIG['N']).sum()} / {len(ELIG)}")

<!--CMT:t1b_ctx-->
**Nhận xét.** (điền sau khi chạy)

<!--CMT:t1_overall-->
**Đánh giá tổng quan bộ dữ liệu.** (điền sau khi chạy)

## 3. Danh mục & backtest
Vũ trụ động: mỗi kỳ chọn N mã có giá trị GD trung vị cao nhất trong cửa sổ, trong số mã (i) có dữ liệu ≥ 95% phiên của cửa sổ, (ii) có giá ở phiên trước ngày tái cân bằng và (iii) **v3:** tỷ lệ phiên r = 0 trong cửa sổ ≤ `ZERO_MAX` (chỉ dùng quá khứ). Mã ngừng GD/hủy niêm yết trong kỳ nắm giữ → lợi suất 0 đến kỳ tái cân bằng sau (giữ tiền mặt).

In [ ]:
def gmv_unconstrained(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def gmv_long_only(S, iters=3000):
    # FISTA + chiếu lên đơn hình (khớp SLSQP, nhanh hơn)
    n = len(S)
    def proj(v):
        u = np.sort(v)[::-1]; css = np.cumsum(u) - 1
        k = np.nonzero(u - css / np.arange(1, n + 1) > 0)[0][-1]; return np.maximum(v - css[k] / (k + 1), 0)
    L = 2 * np.linalg.eigvalsh(S)[-1]; w = y = np.full(n, 1 / n); t = 1.0
    for _ in range(iters):
        wn = proj(y - 2 * S @ y / L); tn = (1 + np.sqrt(1 + 4*t*t)) / 2
        y = wn + (t - 1) / tn * (wn - w)
        if np.abs(wn - w).max() < 1e-9: return wn
        w, t = wn, tn
    return w

PERIODS = [("2016-17 Ổn định", "2016-01-01", "2017-12-31"), ("2018-19 Điều chỉnh", "2018-01-01", "2019-12-31"),
           ("2020Q1 COVID", "2020-01-01", "2020-03-31"), ("2020-21 Bùng nổ", "2020-04-01", "2021-12-31"),
           ("2022 Sụt giảm", "2022-01-01", "2022-12-31"), ("2023-24 Phục hồi", "2023-01-01", "2024-12-31"),
           ("2025-26", "2025-01-01", "2026-12-31")]
ULOG = []   # log universe mỗi kỳ (v3: có sẵn trong run_backtest)

def run_backtest(P, V, estimators, cfg, long_only):
    N, W, REB = cfg["N"], cfg["WINDOW"], cfg["REB"]
    R = np.log(P).diff(); R = np.expm1(R.mask(R.abs() > cfg["MAX_ABS_RET"], 0.0))
    dates = R.index; names = list(estimators)
    rets = {k: [] for k in names}; idx = []; reb_dates = []
    Wp = {k: pd.Series(dtype=float) for k in names}
    st = {k: dict(turn=[], neff=[], wmax=[], short=[]) for k in names}
    for s in range(W + 1, len(dates) - 1, REB):
        win = R.iloc[s - W:s]; ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna()
        ok &= (win == 0).mean() <= cfg.get("ZERO_MAX", 1.0)          # v3: lọc mã đứng giá (cửa sổ quá khứ)
        if ok.sum() < N: continue
        tick = V.iloc[s - W:s].loc[:, ok].median().sort_values(ascending=False).index[:N]
        ULOG.append(dict(date=dates[s], n_eligible=int(ok.sum()), tickers=list(tick)))
        X = win[tick].fillna(0).values; Y = R.iloc[s:s + REB][tick].fillna(0).values
        idx.extend(dates[s:s + REB]); reb_dates.append(dates[s])
        for k in names:
            S = estimators[k](X)
            w = gmv_long_only(S) if long_only else gmv_unconstrained(S)
            ws = pd.Series(w, index=tick)
            if len(Wp[k]):
                u = ws.index.union(Wp[k].index)
                st[k]["turn"].append((ws.reindex(u).fillna(0) - Wp[k].reindex(u).fillna(0)).abs().sum())
            Wp[k] = ws; rets[k].extend(Y @ w)
            st[k]["neff"].append(1 / (w**2).sum()); st[k]["wmax"].append(w.max()); st[k]["short"].append(np.clip(-w, 0, None).sum())
    out = {}
    for k in names:
        r = pd.Series(rets[k], index=idx); rn = r.copy()
        rn.loc[reb_dates[1:]] -= cfg["COST"] * np.array(st[k]["turn"])
        out[k] = dict(r=r, r_net=rn, m={
            "Vol %": r.std() * np.sqrt(252) * 100, "LN/năm %": r.mean() * 252 * 100,
            "Sharpe": r.mean() / r.std() * np.sqrt(252), "Sharpe sau phí": rn.mean() / rn.std() * np.sqrt(252),
            "Turnover/kỳ": np.mean(st[k]["turn"]), "Số mã hiệu dụng": np.mean(st[k]["neff"]),
            "Tỷ trọng max %": np.mean(st[k]["wmax"]) * 100, "Short %": np.mean(st[k]["short"]) * 100})
    return out

def block_bootstrap_ci(ra, rb, block=21, B=2000, seed=0):
    # CI 95% cho Vol(a) − Vol(b), %/năm. (T3: bổ sung kiểm định Ledoit–Wolf 2011)
    rng = np.random.default_rng(seed); n = len(ra); nb = n // block; d = np.empty(B)
    for i in range(B):
        ix = (rng.integers(0, n - block, nb)[:, None] + np.arange(block)).ravel()
        d[i] = (ra[ix].std() - rb[ix].std()) * np.sqrt(252) * 100
    return np.percentile(d, [2.5, 97.5])

def summarize(out, pairs):
    tab = pd.DataFrame({k: o["m"] for k, o in out.items()}).T
    per = pd.DataFrame({k: {p: o["r"][s:e].std() * np.sqrt(252) * 100 for p, s, e in PERIODS if len(o["r"][s:e]) > 20}
                        for k, o in out.items()}).T
    ci = pd.DataFrame({f"{a} vs {b}": block_bootstrap_ci(out[a]["r"].values, out[b]["r"].values)
                       for a, b in pairs if a in out and b in out}, index=["CI 2.5%", "CI 97.5%"]).T
    ci["Có ý nghĩa"] = (ci["CI 97.5%"] < 0) | (ci["CI 2.5%"] > 0)
    return tab, per, ci

## 4. Chạy chính
Cấu hình chuẩn v3 (ô CONFIG, có lọc đứng giá ZERO_MAX = 20%). 11 bộ ước lượng × 2 chế độ. EW-WeSpeR chậm nhất (tối ưu phổ quần thể mỗi kỳ, ~1–4 s/lần), được cache giữa hai chế độ.

In [ ]:
ESTIMATORS = build_estimators(CONFIG)
PAIRS = [("EW-RIE*", "EW-RIE (q_eff)"), ("EW-RIE*", "EW-NL-LW"), ("EW-RIE*", "EW-WeSpeR"), ("EW-WeSpeR", "EW-NL-LW"), ("EW-RIE*", "NL-LW"),
         ("EW-RIE*", "RIE"), ("NL-LW", "Ledoit-Wolf"), ("VS-NL-LW", "NL-LW")]
RESULTS = {}
for long_only in (False, True):
    mode = "long_only" if long_only else "short_ok"; t0 = time.time()
    out = run_backtest(P, V, ESTIMATORS, CONFIG, long_only)
    RESULTS[mode] = dict(out=out, **dict(zip(["table", "periods", "ci"], summarize(out, PAIRS))))
    print(f"\n===== {mode} | {time.time() - t0:.0f}s =====")
    display(RESULTS[mode]["table"].round(2))
    display(RESULTS[mode]["periods"].round(2))
    display(RESULTS[mode]["ci"].round(3))
print("Newton thất bại:", _STATS, "| WeSpeR dự phòng:", _WSP_SAFE, f"| RMSE log-phân vị WeSpeR TB = {np.mean(_WSP['cost']):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
HL = {"EW-RIE*": "#1baf7a", "EW-WeSpeR": "#8e44ad", "EW-NL-LW": "#eb6834", "NL-LW": "#2a78d6"}
for a, mode, ttl in [(ax[0], "short_ok", "Cho phép bán khống"), (ax[1], "long_only", "Chỉ mua")]:
    v = RESULTS[mode]["table"]["Vol %"].sort_values()
    a.barh(v.index, v.values, color=[HL.get(k, "#c9c9c9") for k in v.index])
    a.set_xlim(v.min() - 1, v.max() + 0.5); a.set_title(f"{ttl}: rủi ro ngoài mẫu (%/năm, trục không từ 0)", fontsize=10)
    for i, x in enumerate(v.values): a.text(x, i, f" {x:.2f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

<!--CMT:main_bar-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# Rủi ro trượt 252 phiên và giá trị tích lũy (chỉ mua) cho 4 phương pháp chính
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for k, c in HL.items():
    r = RESULTS["long_only"]["out"][k]["r"]
    ax[0].plot(r.rolling(252).std() * np.sqrt(252) * 100, color=c, lw=1.6, label=k)
    ax[1].plot((1 + RESULTS["long_only"]["out"][k]["r_net"]).cumprod(), color=c, lw=1.6, label=k)
ax[0].set_title("Chỉ mua: Vol trượt 252 phiên (%/năm)", fontsize=10); ax[1].set_title("Chỉ mua: giá trị tích lũy sau phí (1 = đầu kỳ)", fontsize=10)
for a in ax: a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
rv_ = pd.DataFrame({k: RESULTS["long_only"]["out"][k]["r"].rolling(252).std() for k in HL}).dropna()
print("Tỷ lệ cửa sổ 252 phiên EW-RIE* có Vol thấp nhất trong 4 phương pháp:", f"{(rv_.idxmin(axis=1) == 'EW-RIE*').mean():.1%}",
      "| EW-WeSpeR:", f"{(rv_.idxmin(axis=1) == 'EW-WeSpeR').mean():.1%}")

<!--CMT:main_roll-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# Lưu kết quả (tải về hoặc chép vào Drive)
with pd.ExcelWriter(f"{WORKDIR}/results_N{CONFIG['N']}_T{CONFIG['WINDOW']}_h{CONFIG['HL_EW']}.xlsx") as xw:
    for mode, d in RESULTS.items():
        d["table"].round(4).to_excel(xw, sheet_name=f"{mode}_tong_hop")
        d["periods"].round(3).to_excel(xw, sheet_name=f"{mode}_giai_doan")
        d["ci"].round(4).to_excel(xw, sheet_name=f"{mode}_CI")
    pd.Series(CONFIG).to_frame("giá trị").to_excel(xw, sheet_name="CONFIG")
print("Đã lưu vào", WORKDIR)

## 5. Độ nhạy half-life (T6 mở rộng thêm N, T)

In [ ]:
RUN_SENS = False   # đặt True để chạy (~10–20 phút)
if RUN_SENS:
    rows = []
    for hl in (30, 60, 120):
        est = {"EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl), "EW-RIE*": make_ew(ew_rie_corr, hl)}
        for lo in (False, True):
            o = run_backtest(P, V, est, CONFIG, lo)
            l, u = block_bootstrap_ci(o["EW-RIE*"]["r"].values, o["EW-NL-LW"]["r"].values)
            rows.append(dict(hl=hl, mode="long_only" if lo else "short_ok", vol_EWNLLW=o["EW-NL-LW"]["m"]["Vol %"],
                             vol_EWRIE=o["EW-RIE*"]["m"]["Vol %"], CI_lo=l, CI_hi=u))
    display(pd.DataFrame(rows).round(3))

## 6. Việc của nhóm
Dữ liệu: HOSE 2015-01 → 2026-11 (repo `data_HOSE/data_hose`, gồm mã hủy niêm yết), cấu hình chuẩn ở ô CONFIG (N=100, WINDOW=250, REB=21, COST=0,25% một chiều, HL_EW=60, HL_VOL=20, **ZERO_MAX=20%**), SEED=0.

**Quy ước chung cho T2–T6**
- So sánh công bằng: trong mỗi bảng mọi phương pháp dùng **cùng ngày ngoài mẫu, cùng universe, cùng REB/COST/chế độ** (assert trong `run`). Khi các cấu hình có số kỳ khác nhau, so trên **ngày chung**.
- *Gộp* = trước phí; *sau phí* = trừ COST × Σ|Δw| tại ngày tái cân bằng. *Chỉ mua* = `long_only`, *bán khống* = `short_ok`.
- Mọi ma trận hiệp phương sai được kiểm tra đối xứng, bán xác định dương, không NaN. Mọi con số trong phần **Nhận xét** được sinh bằng code từ bảng.
- Bảng lưu ở `./results/*.csv`, hình ở `./figs/*.png`; mỗi backtest được lưu cache ở `./results/cache` (đặt `RECOMPUTE = True` để chạy lại).

In [ ]:
# ---- 6.0 Tiện ích chung cho T1c–T6 (tái dùng run_backtest, summarize, block_bootstrap_ci) ----
from scipy.stats import norm
if importlib.util.find_spec("arch") is None:      # Colab chưa có sẵn `arch` (dùng cho DCC-GARCH ở T3)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"], check=True)
SEED = 0; np.random.seed(SEED)
RESULTS_DIR, FIGS_DIR, CACHE = "./results", "./figs", "./results/cache"
for d in (RESULTS_DIR, FIGS_DIR, CACHE): os.makedirs(d, exist_ok=True)
RECOMPUTE = False                       # True: chạy lại mọi backtest; False: dùng cache results/cache/*.pkl nếu có
CORE, MODES = ["NL-LW", "EW-NL-LW", "EW-RIE*"], {"long_only": True, "short_ok": False}
VN = {"long_only": "Chỉ mua", "short_ok": "Bán khống"}
COL = {"NL-LW": "#2a78d6", "EW-NL-LW": "#eb6834", "EW-RIE*": "#1baf7a", "EW-WeSpeR": "#8e44ad", "EW-NL-LW (v2)": "#eda100"}

TIMES = {}
def check_cov(S, name=""):                      # đối xứng, PSD, không NaN
    ev = np.linalg.eigvalsh((S + S.T) / 2)
    assert np.isfinite(S).all() and np.allclose(S, S.T, rtol=0, atol=1e-10 * np.abs(S).max()) and ev[0] > -1e-10 * ev[-1], name
    return ev
def timed(name, f):
    def g(X):
        t = time.perf_counter(); S = f(X); TIMES.setdefault(name, []).append(time.perf_counter() - t); check_cov(S, name); return S
    return g

def run(est, cfg=None, long_only=True, data=None, tag=""):
    # backtest có cache: khóa = tên bộ ước lượng + cfg + chế độ (+ tag dữ liệu). Trả về out, log universe, thời gian, số lần Newton thất bại.
    cfg = cfg or CONFIG; Pd, Vd = data if data is not None else (P, V)
    f = f"{CACHE}/{hashlib.md5(repr((sorted(est), sorted(cfg.items()), long_only) + ((tag,) if tag else ())).encode()).hexdigest()[:12]}.pkl"
    if os.path.exists(f) and not RECOMPUTE: d = pd.read_pickle(f)
    else:
        TIMES.clear(); ULOG.clear(); f0, n0 = _STATS["fail"], _STATS["total"]
        out = run_backtest(Pd, Vd, {k: timed(k, g) for k, g in est.items()}, cfg, long_only)
        d = dict(out=out, ulog=list(ULOG), sec={k: np.mean(v) for k, v in TIMES.items()},
                 newton=(_STATS["fail"] - f0, _STATS["total"] - n0)); pd.to_pickle(d, f)
    idx = [o["r"].index for o in d["out"].values()]
    assert all(i.equals(idx[0]) for i in idx), "common-sample vi phạm"      # cùng ngày OOS cho mọi phương pháp
    return d

net = lambda o, c: o["r"] - (c / CONFIG["COST"]) * (o["r"] - o["r_net"])   # r_net ở mức phí c (phí không đổi tỷ trọng)
ann = lambda r: (r.std() * np.sqrt(252) * 100, r.mean() / r.std() * np.sqrt(252))
def metrics(o, cost=None, dates=None):
    r, rn = o["r"], net(o, cost or CONFIG["COST"])
    if dates is not None: r, rn = r.loc[dates], rn.loc[dates]
    return {"Vol %": ann(r)[0], "Sharpe": ann(r)[1], "Sharpe sau phí": ann(rn)[1], "LN/năm %": r.mean() * 252 * 100,
            "Turnover/kỳ": o["m"]["Turnover/kỳ"], "Số mã hiệu dụng": o["m"]["Số mã hiệu dụng"], "Số ngày OOS": len(r)}

f2 = lambda x, d=2: f"{x:.{d}f}".replace(".", ",")
pv = lambda p: "< 0,001" if p < 1e-3 else f2(p, 3)
md = lambda s: display(Markdown(s))
def show(df, d=3): display(df.style.format(precision=d, decimal=",", thousands=" ", na_rep="–"))
def save(df, name):
    assert np.isfinite(df.select_dtypes("number").values).all(), f"{name}: NaN/inf"; df.to_csv(f"{RESULTS_DIR}/{name}.csv")
def savefig(fig, name): fig.savefig(f"{FIGS_DIR}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

R_BT = np.log(P).diff(); R_BT = np.expm1(R_BT.mask(R_BT.abs() > CONFIG["MAX_ABS_RET"], 0.0))   # đúng như trong run_backtest
POS = {d: i for i, d in enumerate(R_BT.index)}
BASE = run({k: ESTIMATORS[k] for k in CORE}, CONFIG, True)                 # cấu hình chuẩn, chỉ mua
assert all(np.allclose(BASE["out"][k]["r"].values, RESULTS["long_only"]["out"][k]["r"].values) for k in CORE), "run ≠ §4"
U0 = BASE["ulog"]
ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, arch=__import__("arch").__version__, SEED=SEED, **CONFIG)
json.dump(ENV, open(f"{RESULTS_DIR}/ENV_CONFIG.json", "w"), indent=1)
print(f"Dữ liệu: {P.shape[1]} mã, {P.shape[0]} phiên ({P.index[0].date()} → {P.index[-1].date()}) | OOS: {len(U0)} kỳ, "
      f"{len(BASE['out']['EW-RIE*']['r'])} phiên ({U0[0]['date'].date()} → {BASE['out']['EW-RIE*']['r'].index[-1].date()})")
print(f"Mã đủ điều kiện mỗi kỳ: {min(u['n_eligible'] for u in U0)}–{max(u['n_eligible'] for u in U0)} (N={CONFIG['N']}, ZERO_MAX={CONFIG['ZERO_MAX']:.0%}) | "
      f"run() trùng §4: OK | môi trường: {ENV['python']}, numpy {ENV['numpy']}, pandas {ENV['pandas']}, arch {ENV['arch']}")

### T1c – Survivorship bias: dữ liệu đầy đủ vs chỉ mã còn niêm yết
**Câu hỏi:** nếu chỉ dùng mã còn niêm yết đến 11/2026 (như dữ liệu v2 lấy danh sách mã hiện tại), kết quả lệch bao nhiêu?
So sánh cùng cấu hình chuẩn: (i) đầy đủ (gồm mã đã rời HOSE), (ii) chỉ mã còn niêm yết HOSE (loại mọi mã có ngày hủy niêm yết).

In [ ]:
SURV = [c for c in P.columns if c not in DELISTED]; DATA_SURV = (P[SURV], V[SURV])
T1C_RUN = {(lab, m): run({k: ESTIMATORS[k] for k in CORE}, CONFIG, lo, data=dat, tag=tg)
           for lab, dat, tg in [("(i) đầy đủ", None, ""), ("(ii) chỉ mã còn niêm yết", DATA_SURV, "survivors")] for m, lo in MODES.items()}
COMMON_S = sorted(set.intersection(*[set(d["out"]["NL-LW"]["r"].index) for d in T1C_RUN.values()]))
T1C = pd.DataFrame([dict(data=lab, mode=m, method=k, **metrics(o, dates=COMMON_S)) for (lab, m), d in T1C_RUN.items() for k, o in d["out"].items()])
save(T1C, "T1c_survivorship")
tb = T1C.pivot_table(index=["mode", "method"], columns="data", values=["Vol %", "LN/năm %", "Sharpe sau phí"])
for c in ("Vol %", "LN/năm %", "Sharpe sau phí"): tb[(c, "Δ (ii) − (i)")] = tb[(c, "(ii) chỉ mã còn niêm yết")] - tb[(c, "(i) đầy đủ")]
print(f"Bảng T1c – {len(COMMON_S)} ngày OOS chung; {len(DELISTED)} mã đã rời HOSE bị loại ở (ii)"); show(tb.sort_index(axis=1).rename(index=VN, level=0))
# tỷ trọng mã "sau này hủy niêm yết" trong universe đầy đủ mỗi kỳ và mã hủy ngay trong kỳ nắm giữ
sh = pd.DataFrame([dict(date=u["date"], later=np.mean([t in DELISTED for t in u["tickers"]]),
                        within=np.mean([t in LAST_DELIST.index and u["date"] <= LAST_DELIST[t] <= u["date"] + pd.Timedelta(days=31) for t in u["tickers"]]))
                   for u in T1C_RUN[("(i) đầy đủ", "long_only")]["ulog"]]).set_index("date")
fig, ax = plt.subplots(1, 2, figsize=(14, 3.9))
ax[0].plot(sh.later * 100, color="#eb6834", lw=2, label="% universe là mã sau này rời HOSE")
ax[0].bar(sh.index, sh.within * 100, width=20, color="#52514e", label="% universe hủy niêm yết trong kỳ nắm giữ")
ax[0].set_title("Thành phần universe (N=100) – dữ liệu đầy đủ", fontsize=10); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
x = np.arange(len(CORE)); L_ = T1C[T1C["mode"] == "long_only"].pivot(index="method", columns="data", values="Vol %").reindex(CORE)
for j, (c, col) in enumerate(zip(L_.columns, ["#2a78d6", "#eda100"])): ax[1].bar(x + (j - .5) * .38, L_[c], .36, color=col, label=c)
for j, c in enumerate(L_.columns):
    for i, v in enumerate(L_[c]): ax[1].text(i + (j - .5) * .38, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax[1].set_xticks(x, CORE); ax[1].set_ylim(L_.values.min() - 1, L_.values.max() + .6); ax[1].legend(fontsize=8)
ax[1].set_title("Chỉ mua: Vol %/năm – đầy đủ vs survivorship (trục không từ 0)", fontsize=10)
plt.tight_layout(); savefig(fig, "T1c_survivorship")
print(f"Universe đầy đủ: TB {sh.later.mean():.1%} mã sau này rời HOSE (max {sh.later.max():.1%}); TB {sh.within.mean():.2%} hủy trong kỳ nắm giữ")

<!--CMT:t1c-->
**Nhận xét.** (điền sau khi chạy)

### T2 – Độ vững: chi phí giao dịch, tần suất tái cân bằng, phiên không giao dịch
**Câu hỏi:** hiệu quả của EW-RIE* có còn khi thay đổi điều kiện vận hành danh mục?
- **T2a** lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%} (9 kịch bản × 2 chế độ).
- **T2b** phiên lợi suất 0 (mã đứng giá): không lọc vs lọc > 20% (**chuẩn v3**) vs lọc > 10% phiên r = 0 trong cửa sổ quá khứ.
- **T2c** quy ước phí: COST một chiều vs phí thực tế mua 0,15% / bán 0,25% (thuế bán 0,1%); unit test turnover × cost.

In [ ]:
# ---- T2: Độ vững ----
# T2a. Lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%} cho NL-LW, EW-NL-LW, EW-RIE*, cả hai chế độ.
# COST không làm đổi tỷ trọng ⇒ mỗi (REB, chế độ) chạy 1 backtest; Sharpe sau phí ở từng COST tính lại từ chuỗi turnover (kiểm ở T2c).
REBS, COSTS = (5, 21, 63), (0.0015, 0.0025, 0.004)
T2 = {(reb, m): run({k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, REB=reb), lo) for reb in REBS for m, lo in MODES.items()}
T2A = pd.DataFrame([dict(mode=m, REB=reb, COST=c, method=k, **metrics(o, c), **{"Turnover/năm": o["m"]["Turnover/kỳ"] * 252 / reb})
                    for (reb, m), d in T2.items() for k, o in d["out"].items() for c in COSTS])
save(T2A, "T2a_reb_cost_grid")
g = T2A.sort_values("COST").groupby(["method", "mode", "REB"])
assert g["Sharpe sau phí"].apply(lambda s: (s.diff().dropna() <= 1e-12).all()).all(), "COST tăng mà Sharpe sau phí tăng?"

tab = T2A[T2A.COST == CONFIG["COST"]].pivot_table(index=["mode", "REB"], columns="method",
        values=["Vol %", "Sharpe sau phí", "Turnover/năm"])[["Vol %", "Sharpe sau phí", "Turnover/năm"]].reindex(columns=CORE, level=1)
print(f"Bảng T2a – COST = {CONFIG['COST']:.2%} (Vol %/năm gộp; Sharpe sau phí; turnover năm hóa = Σ|Δw|/kỳ × 252/REB)")
show(tab.rename(index=VN, level=0))

fig, ax = plt.subplots(2, 3, figsize=(13, 6.3)); LOD = T2A[T2A["mode"] == "long_only"]
for j, k in enumerate(CORE):
    for i, (col, cm) in enumerate([("Vol %", "Blues"), ("Sharpe sau phí", "Greens")]):
        M = LOD[LOD.method == k].pivot(index="REB", columns="COST", values=col); lo_, hi_ = LOD[col].min(), LOD[col].max()
        ax[i, j].imshow(M.values, cmap=cm, aspect="auto", vmin=lo_, vmax=hi_)
        ax[i, j].set_xticks(range(3), [f"{c:.2%}" for c in M.columns]); ax[i, j].set_yticks(range(3), M.index)
        for (y, x), v in np.ndenumerate(M.values):
            ax[i, j].text(x, y, f2(v), ha="center", va="center", fontsize=9, color="white" if v > (lo_ + hi_) / 2 else "black")
        ax[i, j].set_title(f"{k} — {col}", fontsize=10); ax[i, j].set_xlabel("COST (1 chiều)"); ax[i, j].set_ylabel("REB (phiên)")
fig.suptitle("T2a – chỉ mua: Vol (gộp) và Sharpe sau phí theo REB × COST (cùng thang màu giữa 3 phương pháp)"); plt.tight_layout(); savefig(fig, "T2a_heatmap")

a = T2A[T2A.COST == CONFIG["COST"]].set_index(["mode", "REB", "method"])
win = lambda m: (T2A[T2A["mode"] == m].pivot_table(index=["REB", "COST"], columns="method", values="Sharpe sau phí")
                 .pipe(lambda x: (x["NL-LW"] > x["EW-RIE*"]).sum()))
low = sum(a.loc[(m, r)]["Vol %"].idxmin() == "EW-RIE*" for m in MODES for r in REBS)

<!--CMT:t2a-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T2b. Phiên lợi suất 0. Code gốc: phiên không khớp lệnh vẫn có giá ⇒ r = 0; NaN trong cửa sổ → fillna(0).
# So sánh (i) gốc với (ii) loại khỏi universe các mã có tỷ lệ r=0 trong CỬA SỔ QUÁ KHỨ > 20% hoặc > 10% (không dùng tương lai).
def zero_share(ulog):    # tỷ lệ r=0 trung bình của universe đã chọn (đúng dữ liệu đưa vào estimator)
    return np.mean([(R_BT.iloc[POS[u["date"]] - CONFIG["WINDOW"]:POS[u["date"]]][u["tickers"]].fillna(0) == 0).values.mean() for u in ulog])
VARS = {"(i) không lọc": 1.0, "(ii) lọc r=0 > 20% (chuẩn v3)": 0.20, "(iii) lọc r=0 > 10%": 0.10}
T2B_RUN = {(v, m): run({k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, ZERO_MAX=z), lo)
           for v, z in VARS.items() for m, lo in MODES.items()}
COMMON = sorted(set.intersection(*[set(d["out"]["NL-LW"]["r"].index) for d in T2B_RUN.values()]))   # kỳ thiếu mã bị bỏ ⇒ so trên ngày chung
T2B = pd.DataFrame([dict(variant=v, mode=m, method=k, **metrics(o, dates=COMMON), **{"Số kỳ hợp lệ": len(d["ulog"]),
                    "Mã đủ ĐK (TB)": np.mean([u["n_eligible"] for u in d["ulog"]]), "Mã đủ ĐK (min)": min(u["n_eligible"] for u in d["ulog"]),
                    "% r=0 trong universe": 100 * zero_share(d["ulog"])}) for (v, m), d in T2B_RUN.items() for k, o in d["out"].items()])
save(T2B, "T2b_zero_return")
print(f"Bảng T2b – Vol %/năm trên {len(COMMON)} ngày OOS chung ({COMMON[0].date()} → {COMMON[-1].date()}); tổng số kỳ chuẩn: {len(U0)}")
tb = T2B.pivot_table(index="variant", columns=["mode", "method"], values="Vol %").reindex(columns=CORE, level=1)
info = T2B[T2B["mode"] == "long_only"].groupby("variant")[["Số kỳ hợp lệ", "Mã đủ ĐK (TB)", "Mã đủ ĐK (min)", "% r=0 trong universe"]].first()
show(pd.concat([tb.rename(columns=VN, level=0), pd.concat({"Universe": info}, axis=1)], axis=1).loc[list(VARS)], 2)
z = tb["long_only"]; base = z.loc[list(VARS)[0]]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, m in zip(ax, MODES):
    d = tb[m].loc[list(VARS)]; x = np.arange(len(CORE))
    for j, (vn, col) in enumerate(zip(VARS, ["#9aa0a6", "#1baf7a", "#eda100"])):
        a.bar(x + (j - 1) * .27, d.loc[vn, CORE], .25, color=col, label=vn)
        for i, v in enumerate(d.loc[vn, CORE]): a.text(i + (j - 1) * .27, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
    a.set_xticks(x, CORE); a.set_ylim(tb.values.min() - 1.5, tb.values.max() + .8); a.set_title(f"{VN[m]}: Vol %/năm theo mức lọc đứng giá", fontsize=10); a.grid(axis="y", alpha=.3)
ax[0].legend(fontsize=8); plt.tight_layout(); savefig(fig, "T2b_zero_filter")

<!--CMT:t2b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T2c. Quy ước chi phí. run_backtest: phí kỳ = COST × Σ|w_mới − w_cũ| ⇒ COST là phí MỘT CHIỀU áp cho cả mua lẫn bán.
# HOSE: phí môi giới ~0,15% mỗi chiều + thuế bán 0,1% ⇒ bất đối xứng (mua 0,15%, bán 0,25%).
def trade_cost(w_new, w_old, c_buy, c_sell):
    u = w_new.index.union(w_old.index); d = w_new.reindex(u, fill_value=0) - w_old.reindex(u, fill_value=0)
    return c_buy * d.clip(lower=0).sum() + c_sell * (-d).clip(lower=0).sum()
rng = np.random.default_rng(SEED)
for _ in range(200):     # unit test: có mã ra/vào universe và tỷ trọng âm (bán khống), Σw = 1 ở cả hai kỳ
    a_ = pd.Series(rng.normal(size=6), index=list("ABCDEF")); a_ /= a_.sum(); b_ = pd.Series(rng.dirichlet(np.ones(4)), index=list("DEFG"))
    u = a_.index.union(b_.index); turn = (a_.reindex(u, fill_value=0) - b_.reindex(u, fill_value=0)).abs().sum()
    assert np.isclose(trade_cost(a_, b_, .0025, .0025), .0025 * turn) and np.isclose(trade_cost(a_, b_, .0015, .0025), .002 * turn)
o_new = run({"NL-LW": est_nllw}, dict(CONFIG, REB=63, COST=0.004))["out"]["NL-LW"]          # chạy lại thật với COST khác
assert np.allclose(net(T2[63, "long_only"]["out"]["NL-LW"], 0.004).values, o_new["r_net"].values, atol=1e-14)
print("Unit test 200/200 pass: Σw=1 ⇒ giá trị mua = giá trị bán = turnover/2 ⇒ phí (mua 0,15%, bán 0,25%) ≡ COST đối xứng 0,20%.")
print("Kiểm chứng: r_net tính lại từ turnover trùng với chạy lại backtest thật (REB=63, COST=0,40%).")
SC = {"Gộp": None, "COST 0,25% (gốc)": .0025, "Mua 0,15% / bán 0,25%": .002}
T2C = pd.DataFrame([dict(mode=VN[m], method=k, scen=sc, Sharpe=ann(o["r"] if c is None else net(o, c))[1])
                    for m in MODES for k, o in T2[21, m]["out"].items() for sc, c in SC.items()]).pivot(index="method", columns=["mode", "scen"], values="Sharpe")
T2C = T2C.reindex(index=CORE, columns=pd.MultiIndex.from_product([VN.values(), SC]))
save(T2C, "T2c_cost_convention"); print("Sharpe (REB = 21):"); show(T2C)

<!--CMT:t2c-->
**Nhận xét.** (điền sau khi chạy)

### T3 – Đối chứng & kiểm định
**Câu hỏi:** EW-RIE* có vượt các đối chứng mạnh trong cùng điều kiện, và chênh lệch có vượt dao động ngẫu nhiên của mẫu?
- **T3a** đối chứng mới: **EW-WeSpeR (v3)**, 1/N (`est_equal_weight_proxy`, chiến lược phân bổ – không phải bộ ước lượng), PCA k nhân tố (k ∈ {1,3,5}; PCA-auto chọn k chỉ trong cửa sổ huấn luyện), DCC-GARCH(1,1) (GARCH đơn biến bằng `arch` + DCC đầy đủ N×N, (a, b) ước lượng bằng composite likelihood).
- **T3b** bảng benchmark: Vol, Sharpe trước/sau phí, turnover, số mã hiệu dụng, thời gian chạy; Vol theo giai đoạn.
- **T3c** kiểm định phương sai Ledoit–Wolf (2011): Δ = log σ²(EW-RIE*) − log σ²(đối chứng), HAC và studentized circular block bootstrap (block 10/21/42); Sharpe: Jobson–Korkie–Memmel; Holm cho 5 so sánh mỗi chế độ.
- **T3d** hình: Vol kèm CI 95%, heatmap p-value.

In [ ]:
# ---- T3: Đối chứng & kiểm định ----
# T3a. Đối chứng mới: 1/N, mô hình nhân tố PCA (k ∈ {1,3,5} và tự chọn k trong cửa sổ huấn luyện), DCC-GARCH(1,1).
from arch import arch_model
from scipy.optimize import minimize

def est_equal_weight_proxy(X):
    # 1/N: dùng ma trận đơn vị -> GMV = 1/N (chiến lược phân bổ, KHÔNG phải bộ ước lượng hiệp phương sai)
    return np.eye(X.shape[1])

def est_pca(k):
    # Σ = B Bᵀ + diag(ψ): k thành phần chính của hiệp phương sai mẫu, phần dư chéo (đối xứng, PSD theo cấu trúc)
    def est(X):
        S = np.cov(X, rowvar=False); lam, Q = np.linalg.eigh(S); B = Q[:, -k:] * np.sqrt(lam[-k:])
        return B @ B.T + np.diag(np.maximum(np.diag(S) - (B**2).sum(1), 1e-10))
    return est

PCA_K = []
def est_pca_auto(X, ks=(1, 3, 5), split=0.7):
    # chọn k CHỈ trong cửa sổ huấn luyện: ước lượng trên 70% đầu, đo phương sai GMV trên 30% cuối, rồi ước lượng lại trên cả cửa sổ
    n = int(len(X) * split)
    k = min(ks, key=lambda k: np.var(X[n:] @ gmv_unconstrained(est_pca(k)(X[:n]))))
    PCA_K.append(k); return est_pca(k)(X)

DCC_LOG, _DCC_CACHE = [], {}
def _dcc_ncl(p, Z, Qbar):
    # −composite log-lik (Engle, Shephard & Sheppard 2008): tổng log-lik 2×2 của các cặp kề nhau (i, i+1).
    # Full QML với N=100 vừa chậm vừa kéo a → 0 (thiên lệch đã biết khi N lớn) nên dùng CL.
    a, b = p
    if a < 0 or b < 0 or a + b >= 0.999: return 1e10
    z1, z2 = Z[:, :-1], Z[:, 1:]; q11, q22, q12 = np.diag(Qbar)[:-1], np.diag(Qbar)[1:], np.diag(Qbar, 1)
    c11, c22, c12 = q11.copy(), q22.copy(), q12.copy(); nll = 0.0
    for t in range(len(Z)):
        if t:
            c11 = (1-a-b)*q11 + a*z1[t-1]**2 + b*c11; c22 = (1-a-b)*q22 + a*z2[t-1]**2 + b*c22
            c12 = (1-a-b)*q12 + a*z1[t-1]*z2[t-1] + b*c12
        rho = c12 / np.sqrt(c11 * c22); d = 1 - rho**2
        nll += 0.5 * np.sum(np.log(d) + (z1[t]**2 + z2[t]**2 - 2*rho*z1[t]*z2[t]) / d)
    return nll

def est_dcc(X, h=21):
    # DCC-GARCH(1,1) (Engle 2002): bước 1 GARCH(1,1) đơn biến (`arch`) cho từng mã; bước 2 (a, b) bằng composite
    # likelihood + correlation targeting; ma trận Q_t đầy đủ N×N. Dự báo: D = căn phương sai GARCH trung bình h phiên tới, R = R_{T+1|T}; Σ = D R D.
    key = hash(X.tobytes())
    if key in _DCC_CACHE: return _DCC_CACHE[key]               # cùng cửa sổ ở long_only/short_ok → không ước lượng lại
    T, N = X.shape; Z = np.empty_like(X); hv = np.empty(N); bad = 0
    for i in range(N):
        try:
            r = arch_model(100 * X[:, i], mean="Constant", vol="GARCH", p=1, q=1, rescale=False).fit(disp="off", show_warning=False)
            ok = r.convergence_flag == 0 and r.params["alpha[1]"] + r.params["beta[1]"] < 1
        except Exception:
            ok = False
        if ok:
            Z[:, i] = r.std_resid; hv[i] = r.forecast(horizon=h, reindex=False).variance.values[-1].mean() / 1e4
        else:                                                   # dự phòng: phương sai mẫu (đếm vào DCC_LOG)
            bad += 1; Z[:, i] = (X[:, i] - X[:, i].mean()) / X[:, i].std(); hv[i] = X[:, i].var()
    Qbar = Z.T @ Z / T
    res = minimize(_dcc_ncl, x0=[0.02, 0.95], args=(Z, Qbar), method="Nelder-Mead", options=dict(xatol=1e-4, fatol=1e-3, maxiter=200))
    a, b = res.x if res.fun < 1e9 else (0.0, 0.0)
    Qt = Qbar.copy()
    for t in range(1, T + 1): Qt = (1 - a - b) * Qbar + a * np.outer(Z[t-1], Z[t-1]) + b * Qt    # Q_{T+1}
    d = 1 / np.sqrt(np.diag(Qt)); sd = np.sqrt(hv)
    S = (Qt * np.outer(d, d)) * np.outer(sd, sd)
    DCC_LOG.append(dict(a=a, b=b, garch_fail=bad, dcc_ok=bool(res.success))); _DCC_CACHE[key] = S
    return S

In [ ]:
# T3b. Bảng benchmark: cùng ngày OOS, cùng universe, REB=21, COST=0,25%, hai chế độ.
T3_EST = {"1/N": est_equal_weight_proxy, **{f"PCA-k{k}": est_pca(k) for k in (1, 3, 5)}, "PCA-auto": est_pca_auto,
          "DCC-GARCH(1,1)": est_dcc, "EW-WeSpeR": ESTIMATORS["EW-WeSpeR"], **{k: ESTIMATORS[k] for k in CORE}}
T3R = {m: run(T3_EST, CONFIG, lo) for m, lo in MODES.items()}
T3B = pd.DataFrame([dict(mode=m, method=k, **metrics(o), **{"Tỷ trọng max %": o["m"]["Tỷ trọng max %"], "Giây/lần ước lượng": T3R["long_only"]["sec"][k]})
                    for m, d in T3R.items() for k, o in d["out"].items()])
save(T3B, "T3b_benchmark")
for m in MODES: assert T3B[T3B["mode"] == m].set_index("method")["Turnover/kỳ"].idxmin() == "1/N"   # sanity: 1/N ít giao dịch nhất
tb = T3B.pivot(index="method", columns="mode", values=["Vol %", "Sharpe", "Sharpe sau phí", "Turnover/kỳ", "Số mã hiệu dụng"])
tb.columns = tb.columns.swaplevel(); tb = tb[[c for m in MODES for c in tb.columns if c[0] == m]].sort_values(("long_only", "Vol %"))
tb["Giây/lần ước lượng"] = T3B.drop_duplicates("method").set_index("method")["Giây/lần ước lượng"]
print("Bảng T3b – benchmark (xếp theo Vol chỉ mua; Vol %/năm gộp; Sharpe sau phí trừ COST × turnover)"); show(tb.rename(columns=VN, level=0))
P3 = pd.concat({VN[m]: summarize(d["out"], [])[1] for m, d in T3R.items()}); save(P3, "T3b_periods")
print("Vol %/năm theo giai đoạn thị trường:"); show(P3, 2)
if len(DCC_LOG) >= len(U0):   # log chỉ có khi vừa tính mới (không dùng cache) → lưu CSV; kỳ đầu tiên của chế độ chỉ mua
    pd.DataFrame(DCC_LOG[:len(U0)]).assign(pca_auto_k=PCA_K[:len(U0)]).to_csv(f"{RESULTS_DIR}/T3_dcc_pca_log.csv", index=False)
dl = pd.read_csv(f"{RESULTS_DIR}/T3_dcc_pca_log.csv"); pk = dl.pca_auto_k
v = tb[("long_only", "Vol %")]

<!--CMT:t3b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T3c. Kiểm định: phương sai Ledoit–Wolf (2011) – HAC và studentized circular block bootstrap; Sharpe Jobson–Korkie–Memmel;
# so với block bootstrap CI có sẵn (block_bootstrap_ci), block ∈ {10, 21, 42}; Holm cho 4 so sánh mỗi chế độ.
def hac(u, L=None):
    # HAC Newey–West (Bartlett), L = ⌊4(T/100)^(2/9)⌋. (LW 2008/2011 dùng QS/Parzen + prewhitening – chưa làm.)
    T = len(u); L = L or int(4 * (T / 100) ** (2 / 9)); S = u.T @ u / T
    for l in range(1, L + 1): G = u[l:].T @ u[:-l] / T; S += (1 - l / (L + 1)) * (G + G.T)
    return S

def _lw_stat(y, Psi=None, b=None):
    # Δ = log σ²_a − log σ²_b (LW 2011, delta method); y_t = (r_a, r_b, r_a², r_b²)
    T = len(y); m = y.mean(0); va, vb = m[2] - m[0]**2, m[3] - m[1]**2
    g = np.array([-2 * m[0] / va, 2 * m[1] / vb, 1 / va, -1 / vb])
    if Psi is None:   # Ψ của bootstrap theo khối (LW 2008, công thức (9))
        z = (y - m)[:T // b * b].reshape(-1, b, 4).sum(1) / np.sqrt(b); Psi = z.T @ z / len(z)
    return np.log(va) - np.log(vb), np.sqrt(g @ Psi @ g / T)

def lw_var_test(ra, rb, block=21, M=2000, seed=SEED):
    # p-value HAC (chuẩn) và p-value studentized circular block bootstrap (LW 2008 Remark 3.2, áp cho phương sai như LW 2011)
    y = np.column_stack([ra, rb, ra**2, rb**2]); T = len(y)
    d, se = _lw_stat(y, hac(y - y.mean(0))); p_hac = 2 * norm.sf(abs(d / se))
    rng = np.random.default_rng(seed); l = T // block; stats = np.empty(M)
    for i in range(M):
        ix = ((rng.integers(0, T, l)[:, None] + np.arange(block)) % T).ravel()      # khối vòng
        ds, ses = _lw_stat(y[ix], b=block); stats[i] = abs(ds - d) / ses
    return d, se, p_hac, (np.sum(stats >= abs(d) / se) + 1) / (M + 1)

def jkm_test(ra, rb):
    # Jobson–Korkie (1981) với hiệu chỉnh Memmel (2003); giả định iid chuẩn
    T = len(ra); m1, m2 = ra.mean(), rb.mean(); s1, s2 = ra.std(ddof=1), rb.std(ddof=1); s12 = np.cov(ra, rb)[0, 1]
    th = (2 * s1**2 * s2**2 - 2 * s1 * s2 * s12 + .5 * m1**2 * s2**2 + .5 * m2**2 * s1**2 - m1 * m2 / (s1 * s2) * s12**2) / T
    z = (s2 * m1 - s1 * m2) / np.sqrt(th); return z, 2 * norm.sf(abs(z))

# unit test nhanh: gần trùng ⇒ p lớn; gấp đôi biến động ⇒ p≈0; JKM phản đối xứng, Sharpe chênh lớn ⇒ p≈0
# (JKM suy biến θ=0 khi 2 chuỗi tương quan hoàn hảo – không dùng trường hợp đó)
_r = np.random.default_rng(SEED).standard_normal(1500) * .01
assert lw_var_test(_r, _r + 1e-5 * np.random.default_rng(1).standard_normal(1500), M=200)[3] > .1
assert lw_var_test(_r, 2 * _r[::-1], M=200)[2] < 1e-6                                   # Δ = −log 4
_q = np.random.default_rng(2).standard_normal(1500) * .01
assert np.isclose(jkm_test(_r, _q)[0], -jkm_test(_q, _r)[0]) and jkm_test(_r + .003, _q - .003)[1] < 1e-6

BLOCKS, rows = (10, 21, 42), []
for mode in MODES:
    out = T3R[mode]["out"]
    for b_ in ("NL-LW", "EW-NL-LW", "EW-WeSpeR", "1/N", "PCA-auto"):   # đối chứng định trước, không chọn hậu kiểm
        ra, rb = out["EW-RIE*"]["r"].values, out[b_]["r"].values
        na, nb = (net(out[k], CONFIG["COST"]).values for k in ("EW-RIE*", b_))
        row = dict(mode=mode, pair=f"EW-RIE* vs {b_}", **{"ΔVol (pp)": (ra.std() - rb.std()) * np.sqrt(252) * 100})
        for bl in BLOCKS:
            d, se, p_hac, p_bt = lw_var_test(ra, rb, block=bl)
            lo_, hi_ = block_bootstrap_ci(ra, rb, block=bl)
            row.update({"Δlog σ²": d, "p LW-HAC": p_hac, f"p LW-boot b={bl}": p_bt, f"CI lo b={bl}": lo_, f"CI hi b={bl}": hi_})
        row.update({"p JKM gross": jkm_test(ra, rb)[1], "p JKM sau phí": jkm_test(na, nb)[1],
                    "ΔSharpe gross": ann(pd.Series(ra))[1] - ann(pd.Series(rb))[1]})
        rows.append(row)
T3C = pd.DataFrame(rows)
# multiple comparisons: Holm trên 5 so sánh trong mỗi chế độ (p LW-boot b=21)
T3C["p Holm (LW-boot b=21)"] = T3C.groupby("mode")["p LW-boot b=21"].transform(
    lambda p: np.minimum(1, np.maximum.accumulate(np.sort(p.values) * (len(p) - np.arange(len(p)))))[np.argsort(np.argsort(p.values))])
save(T3C, "T3c_tests")
cols = ["ΔVol (pp)", "CI lo b=21", "CI hi b=21", "p LW-HAC", "p LW-boot b=10", "p LW-boot b=21", "p LW-boot b=42", "p Holm (LW-boot b=21)", "p JKM gross", "p JKM sau phí"]
print("Bảng T3c – EW-RIE* so với đối chứng (ΔVol < 0: EW-RIE* rủi ro thấp hơn; CI = block bootstrap percentile b=21)")
show(T3C.assign(mode=T3C["mode"].map(VN)).set_index(["mode", "pair"])[cols], 4)
t3 = lambda m, b: T3C[(T3C["mode"] == m) & (T3C.pair == f"EW-RIE* vs {b}")].iloc[0]
line = lambda m: "; ".join(f"{b}: {f2(t3(m, b)['ΔVol (pp)'])} pp, p Holm {pv(t3(m, b)['p Holm (LW-boot b=21)'])}" for b in ("NL-LW", "EW-NL-LW", "PCA-auto", "1/N"))
sig = lambda b: all(t3(m, b)["p Holm (LW-boot b=21)"] < .05 for m in MODES)

<!--CMT:t3c-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T3d. Hình – Vol với CI 95% (block bootstrap b=21 của từng chuỗi, tái dùng block_bootstrap_ci với chuỗi 0) + heatmap p-value ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
for a, mode in zip(ax, MODES):
    out = T3R[mode]["out"]; v = pd.Series({k: o["m"]["Vol %"] for k, o in out.items()}).sort_values()
    ci = np.array([block_bootstrap_ci(out[k]["r"].values, np.zeros(len(out[k]["r"]))) for k in v.index])
    a.barh(v.index, v.values, xerr=[v.values - ci[:, 0], ci[:, 1] - v.values], capsize=3, height=.6,
           color=[COL["EW-RIE*"] if k == "EW-RIE*" else "#9aa0a6" for k in v.index], error_kw=dict(lw=1, ecolor="#52514e"))
    for i, x in enumerate(v.values): a.text(ci[i, 1] + .2, i, f"{x:.2f}", va="center", fontsize=8)
    a.set_title(f"{VN[mode]}: Vol ngoài mẫu %/năm (gross), CI 95% block bootstrap b=21", fontsize=10); a.grid(axis="x", alpha=.3)
ax[0].set_xlim(0, None); plt.tight_layout(); savefig(fig, "T3_benchmark")

pc = [c for c in T3C.columns if c.startswith("p ")]
M = T3C.set_index(["mode", "pair"])[pc]
fig, a = plt.subplots(figsize=(11, 4.2))
im = a.imshow(-np.log10(M.values.clip(1e-4)), cmap="Purples", vmin=0, vmax=4, aspect="auto")
a.set_xticks(range(len(pc)), [c[2:] for c in pc], rotation=30, ha="right", fontsize=8)
a.set_yticks(range(len(M)), [f"{VN[m]} | {p}" for m, p in M.index], fontsize=8)
for (y, x), p in np.ndenumerate(M.values): a.text(x, y, f"{p:.3f}", ha="center", va="center", fontsize=8,
                                                 color="white" if p < 0.01 else "black")
plt.colorbar(im, ax=a, label="−log10 p"); a.set_title("T3 — p-value hai phía (LW 2011 phương sai; JKM Sharpe; Holm trên 5 so sánh/chế độ)", fontsize=10)
plt.tight_layout(); savefig(fig, "T3_pvalue_heatmap")

<!--CMT:t3d-->
**Nhận xét.** (điền sau khi chạy)

### T5 – Vì sao VS giúp trên HOSE?
**Câu hỏi:** chuẩn hóa biến động (VS) có giảm rủi ro không, khi nào, và nhờ cơ chế nào?
- **T5a** ablation chỉ đổi VS: RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW; theo giai đoạn thị trường.
- **T5b** dự báo biến động 21 phiên: EWMA vs cửa sổ phẳng (RMSE, QLIKE) theo giai đoạn.
- **T5c** phổ trị riêng trước/sau VS và condition number của Σ đưa vào GMV.
- VS-EW-RIE* **không** làm ở đây (chưa có biến thể đã kiểm thử; không nhầm với VS-RIE).

In [ ]:
# ---- T5: Vì sao VS giúp trên HOSE? ----
# T5a. Ablation: mỗi cặp chỉ khác bước chuẩn hóa biến động (VS): RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW. Dùng lại lần chạy chính §4 (QUICK=False).
ABL = [("RIE", "VS-RIE"), ("NL-LW", "VS-NL-LW")]
rows, per = [], []
for m in MODES:
    out = RESULTS[m]["out"]
    for b, v in ABL:
        lo_, hi_ = block_bootstrap_ci(out[v]["r"].values, out[b]["r"].values)
        mb, mv = metrics(out[b]), metrics(out[v])
        rows.append({"Chế độ": VN[m], "Cặp": f"{v} vs {b}", "Vol gốc": mb["Vol %"], "Vol VS": mv["Vol %"], "ΔVol (pp)": mv["Vol %"] - mb["Vol %"],
                     "CI lo": lo_, "CI hi": hi_, "Sharpe sau phí gốc": mb["Sharpe sau phí"], "Sharpe sau phí VS": mv["Sharpe sau phí"],
                     "Turnover gốc": mb["Turnover/kỳ"], "Turnover VS": mv["Turnover/kỳ"]})
        per.append((RESULTS[m]["periods"].loc[v] - RESULTS[m]["periods"].loc[b]).rename((VN[m], f"{v} − {b}")))
T5A = pd.DataFrame(rows).set_index(["Chế độ", "Cặp"]); T5P = pd.DataFrame(per).T
save(T5A, "T5a_vs_ablation"); save(T5P, "T5a_vs_ablation_periods")
print("Bảng T5a – tác động của VS (Vol %/năm gộp; CI 95% block bootstrap b=21 của ΔVol)"); show(T5A)
print("ΔVol (pp) do VS theo giai đoạn thị trường (< 0: VS giảm rủi ro):"); show(T5P, 2)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, m in zip(ax, VN.values()):
    d = T5P[m]; x = np.arange(len(d))
    for j, (c, col) in enumerate(zip(d.columns, ["#2a78d6", "#eb6834"])): a.bar(x + (j - .5) * .38, d[c], .36, color=col, label=c)
    a.axhline(0, color="#52514e", lw=.8); a.set_xticks(x, d.index, rotation=20, fontsize=8); a.grid(axis="y", alpha=.3)
    a.set_title(f"{m}: ΔVol do VS theo giai đoạn (pp/năm; < 0 ⇒ VS giảm rủi ro)", fontsize=10)
ax[0].legend(fontsize=8); plt.tight_layout(); savefig(fig, "T5a_vs_periods")
pm = T5P.mean(axis=1); tr = T5A["Turnover VS"] / T5A["Turnover gốc"]

<!--CMT:t5a-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T5b. Dự báo phương sai 21 phiên tới: EWMA (hl ∈ {10, 20, 40}, đúng hàm ewma_standardize mà VS dùng) vs cửa sổ phẳng 63/250 phiên.
# Mẫu: các mã trong universe chuẩn ở mỗi kỳ; dự báo chỉ dùng cửa sổ quá khứ. Mục tiêu RV = trung bình r² của 21 phiên sau.
# Sai số: RMSE của vol năm hóa (%) và QLIKE = log h + RV/h (dạng Patton 2011, xác định cả khi RV = 0).
W, H = CONFIG["WINDOW"], 21; rows = []
for u in U0:
    s = POS[u["date"]]
    if s + H > len(R_BT): continue
    X = R_BT.iloc[s - W:s][u["tickers"]].fillna(0).values; rv = (R_BT.iloc[s:s + H][u["tickers"]].fillna(0).values ** 2).mean(0)
    fc = {**{f"EWMA hl={h}": ewma_standardize(X, h)[1] ** 2 for h in (10, 20, 40)}, **{f"Phẳng {w} phiên": (X[-w:] ** 2).mean(0) for w in (63, W)}}
    per_ = next((p for p, a_, b_ in PERIODS if pd.Timestamp(a_) <= u["date"] <= pd.Timestamp(b_)), None)
    for k, h in fc.items():
        h = np.maximum(h, 1e-12); rows.append(dict(period=per_, model=k, se=np.mean((np.sqrt(h) - np.sqrt(rv)) ** 2), ql=np.mean(np.log(h) + rv / h)))
F = pd.DataFrame(rows); F = pd.concat([F, F.assign(period="Toàn mẫu")])
T5B = F.groupby(["period", "model"], sort=False).agg(RMSE=("se", lambda x: np.sqrt(x.mean() * 252) * 100), QLIKE=("ql", "mean"), n=("se", "size"))
save(T5B, "T5b_vol_forecast")
tbl = T5B["RMSE"].unstack()[["EWMA hl=10", "EWMA hl=20", "EWMA hl=40", "Phẳng 63 phiên", f"Phẳng {W} phiên"]]
tbl.loc["QLIKE (toàn mẫu)"] = T5B.xs("Toàn mẫu")["QLIKE"]
print("Bảng T5b – RMSE dự báo vol 21 phiên (%/năm) theo giai đoạn; dòng cuối: QLIKE toàn mẫu (thấp hơn = tốt hơn)"); show(tbl)
fig, ax = plt.subplots(figsize=(11, 3.6)); M = tbl.drop(index="QLIKE (toàn mẫu)"); x = np.arange(len(M))
for j, (c, col) in enumerate(zip(M.columns, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9aa0a6"])): ax.bar(x + (j - 2) * .16, M[c], .15, color=col, label=c)
ax.set_xticks(x, M.index, rotation=15, fontsize=8); ax.set_ylabel("RMSE vol (%/năm)"); ax.legend(fontsize=8, ncol=3); ax.grid(axis="y", alpha=.3)
ax.set_title("T5b – sai số dự báo vol 21 phiên: EWMA vs cửa sổ phẳng", fontsize=10); plt.tight_layout(); savefig(fig, "T5b_vol_forecast")
fa, cv = M.loc["Toàn mẫu"], M.loc["2020Q1 COVID"]

<!--CMT:t5b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T5c. Phổ trị riêng trước/sau VS (1 kỳ ví dụ) + condition number theo kỳ. "Trước": tương quan mẫu của X; "sau": của Z = X/σ_EWMA
# (bỏ HL_VOL phiên đầu, đúng make_vs). Thêm cond của hiệp phương sai THỰC SỰ đưa vào GMV.
hv = CONFIG["HL_VOL"]; rows = []
winX = lambda u: R_BT.iloc[POS[u["date"]] - W:POS[u["date"]]][u["tickers"]].fillna(0).values
for u in U0:
    X = winX(u); Z = ewma_standardize(X, hv)[0][hv:]
    ex, ez = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False)), np.linalg.eigvalsh(np.corrcoef(Z, rowvar=False))
    r = {"date": u["date"], "cond corr(X)": ex[-1] / ex[0], "cond corr(Z), VS": ez[-1] / ez[0], "λ1/N trước VS": ex[-1] / len(ex), "λ1/N sau VS": ez[-1] / len(ez)}
    for k in ("NL-LW", "VS-NL-LW", "RIE", "VS-RIE"): e = check_cov(ESTIMATORS[k](X), k); r[f"cond Σ {k}"] = e[-1] / e[0]
    rows.append(r)
T5C = pd.DataFrame(rows).set_index("date"); save(T5C, "T5c_condition_numbers")
print("Bảng T5c – condition number và trị riêng lớn nhất (trung bình / trung vị trên các kỳ)"); show(T5C.agg(["mean", "median"]).T, 2)
ex_u = next(u for u in U0 if u["date"] >= pd.Timestamp("2020-03-01")); X = winX(ex_u); Z = ewma_standardize(X, hv)[0][hv:]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for lab, M_, c in [("trước VS: corr(X)", X, "#9aa0a6"), ("sau VS: corr(Z)", Z, "#1baf7a")]:
    ev = np.sort(np.linalg.eigvalsh(np.corrcoef(M_, rowvar=False)))[::-1]; ax[0].plot(np.arange(1, len(ev) + 1), ev, color=c, lw=2, marker="o", ms=3, label=lab)
ax[0].axhline((1 + np.sqrt(CONFIG["N"] / W)) ** 2, ls=":", color="#52514e", label="biên MP λ+ (nhiễu thuần)")
ax[0].set_yscale("log"); ax[0].set_xlabel("thứ hạng trị riêng"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[0].set_title(f"Phổ tương quan, kỳ {ex_u['date'].date()} (N={CONFIG['N']}, T={W})", fontsize=10)
for k, c in [("cond Σ NL-LW", "#2a78d6"), ("cond Σ VS-NL-LW", "#eb6834"), ("cond corr(X)", "#9aa0a6"), ("cond corr(Z), VS", "#1baf7a")]:
    ax[1].plot(T5C.index, T5C[k], color=c, lw=2, label=k)
ax[1].set_yscale("log"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3); ax[1].set_title("Condition number theo kỳ tái cân bằng", fontsize=10)
plt.tight_layout(); savefig(fig, "T5c_spectrum_condition")
c = T5C.mean()

<!--CMT:t5c-->
**Nhận xét.** (điền sau khi chạy)

### T6 – Khái quát: số tài sản N, cửa sổ T, half-life (và trường hợp N > T_eff)
**Câu hỏi:** EW-RIE* có duy trì hiệu quả khi đổi quy mô và độ dài dữ liệu? Đối chứng EW-WeSpeR và EW-NL-LW đã sửa có thay đổi kết luận khi N > T_eff?
Đổi từng yếu tố, giữ nguyên phần còn lại: N ∈ {50, 100, 150}, T ∈ {125, 250, 500}, h ∈ {15, 30, 60, 120}. T_eff = 1/Σw², q_eff = N·Σw² (q_eff > 1 ⇔ N > T_eff: h = 15, 30; N = 150 sát ngưỡng). VN30/HNX: chưa làm (cần lịch sử rổ).

In [ ]:
# ---- T6: Khái quát theo N, cửa sổ T, half-life (đổi TỪNG yếu tố, giữ nguyên phần còn lại) ----
# NL-LW, EW-NL-LW (v3), EW-RIE*, EW-WeSpeR + EW-NL-LW (v2) để đo tác động của bản sửa; cả hai chế độ, so trên NGÀY OOS CHUNG.
T6K = CORE + ["EW-WeSpeR", "EW-NL-LW (v2)"]
def t6_est(cfg):
    E = build_estimators(cfg); E["EW-NL-LW (v2)"] = make_ew(lambda C, w: renorm(nl_lw_S_v2(C, 1 / (w**2).sum())), cfg["HL_EW"])
    return {k: E[k] for k in T6K}
GRID = [("N", v) for v in (50, 100, 150)] + [("WINDOW", v) for v in (125, 250, 500)] + [("HL_EW", v) for v in (15, 30, 60, 120)]
T6R = {(f, v, m): run(t6_est(dict(CONFIG, **{f: v})), dict(CONFIG, **{f: v}), lo) for f, v in GRID for m, lo in MODES.items()}
rows = []
for f in ("N", "WINDOW", "HL_EW"):
    for m in MODES:
        common = sorted(set.intersection(*[set(T6R[f, v, m]["out"]["NL-LW"]["r"].index) for ff, v in GRID if ff == f]))
        for ff, v in [g for g in GRID if g[0] == f]:
            d = T6R[ff, v, m]; o = d["out"]; cfg = dict(CONFIG, **{ff: v}); w = ew_weights(cfg["WINDOW"], cfg["HL_EW"])
            ra, rb = o["EW-RIE*"]["r"].loc[common].values, o["EW-NL-LW"]["r"].loc[common].values; lo_, hi_ = block_bootstrap_ci(ra, rb)
            rc = o["EW-WeSpeR"]["r"].loc[common].values; lo2, hi2 = block_bootstrap_ci(ra, rc)
            rows.append(dict(factor=ff, value=v, mode=m, N=cfg["N"], T=cfg["WINDOW"], h=cfg["HL_EW"], q=cfg["N"] / cfg["WINDOW"],
                             T_eff=1 / (w ** 2).sum(), q_eff=cfg["N"] * (w ** 2).sum(), **{f"Vol {k}": ann(o[k]["r"].loc[common])[0] for k in T6K},
                             **{"Δ": (ra.std() - rb.std()) * np.sqrt(252) * 100, "CI lo": lo_, "CI hi": hi_,
                                "Δ vs WeSpeR": (ra.std() - rc.std()) * np.sqrt(252) * 100, "CI2 lo": lo2, "CI2 hi": hi2, "Số ngày chung": len(common),
                                "Số kỳ": len(d["ulog"]), "giây/lần EW-RIE*": d["sec"].get("EW-RIE*", np.nan), "giây/lần EW-WeSpeR": d["sec"].get("EW-WeSpeR", np.nan),
                                "Newton fail": d["newton"][0], "Newton tổng": d["newton"][1]}))
T6 = pd.DataFrame(rows); save(T6, "T6_generalization")
for (f, m), g in T6.groupby(["factor", "mode"]): assert g["Số ngày chung"].nunique() == 1          # common-sample trong mỗi yếu tố
lab = lambda r: {"N": "N", "WINDOW": "T", "HL_EW": "h"}[r.factor] + f"={r.value}"
T6["cấu hình"] = T6.apply(lab, axis=1)
star = lambda c, l, h: T6.apply(lambda r: f2(r[c], 3) + ("*" if r[h] < 0 or r[l] > 0 else ""), axis=1)
tb = T6.assign(Δ=star("Δ", "CI lo", "CI hi"), **{"Δ vs WeSpeR": star("Δ vs WeSpeR", "CI2 lo", "CI2 hi")}).pivot(
    index="cấu hình", columns="mode", values=["Vol NL-LW", "Vol EW-NL-LW", "Vol EW-WeSpeR", "Vol EW-RIE*", "Δ", "Δ vs WeSpeR"])
L6 = T6[T6["mode"] == "long_only"].set_index("cấu hình")
tb.columns = tb.columns.swaplevel(); tb = tb[[c for m in MODES for c in tb.columns if c[0] == m]].reindex(L6.index)
tb[("", "q_eff")] = L6["q_eff"]; tb[("", "ngày chung")] = L6["Số ngày chung"]
print("Bảng T6 – Vol %/năm (ngày OOS chung trong mỗi yếu tố); Δ = Vol(EW-RIE*) − Vol(EW-NL-LW), Δ vs WeSpeR = Vol(EW-RIE*) − Vol(EW-WeSpeR), pp; * = CI 95% block bootstrap không chứa 0")
show(tb.rename(columns=VN, level=0), 2)
print("Thời gian và độ ổn định số (chỉ mua):"); show(L6[["Số kỳ", "giây/lần EW-RIE*", "giây/lần EW-WeSpeR", "Newton fail", "Newton tổng"]], 3)

fig, ax = plt.subplots(2, 4, figsize=(15, 6.5), sharex="col"); LOD = T6[T6["mode"] == "long_only"]
for j, (ttl, x, d) in enumerate([("N", "N", LOD[LOD.factor == "N"]), ("cửa sổ T", "T", LOD[LOD.factor == "WINDOW"]),
                                 ("q_eff = N·Σw²", "q_eff", LOD), ("half-life h", "h", LOD[LOD.factor == "HL_EW"])]):
    d = d.sort_values(x); ls = "none" if x == "q_eff" else "-"
    for k in T6K: ax[0, j].plot(d[x], d[f"Vol {k}"], marker="o", ls=ls if k != "EW-NL-LW (v2)" else ":", lw=2 if k != "EW-NL-LW (v2)" else 1.2, color=COL[k], label=k)
    ax[1, j].errorbar(d[x], d["Δ"], yerr=[d["Δ"] - d["CI lo"], d["CI hi"] - d["Δ"]], fmt="o", ls=ls, color=COL["EW-NL-LW"], capsize=4, lw=2, label="vs EW-NL-LW")
    ax[1, j].errorbar(d[x] * (1.03 if x != "q_eff" else 1.02), d["Δ vs WeSpeR"], yerr=[d["Δ vs WeSpeR"] - d["CI2 lo"], d["CI2 hi"] - d["Δ vs WeSpeR"]],
                      fmt="s", ls=ls, color=COL["EW-WeSpeR"], capsize=4, lw=2, label="vs EW-WeSpeR")
    ax[1, j].axhline(0, color="#52514e", lw=.8); ax[0, j].set_title(f"Chỉ mua: Vol theo {ttl}", fontsize=10); ax[1, j].set_xlabel(ttl)
    ax[0, j].grid(alpha=.3); ax[1, j].grid(alpha=.3)
    if x == "q_eff": ax[0, j].axvline(1, color="#52514e", ls="--", lw=.8); ax[1, j].axvline(1, color="#52514e", ls="--", lw=.8)
ax[0, 0].set_ylabel("Vol %/năm"); ax[1, 0].set_ylabel("Δ Vol của EW-RIE* (pp), CI 95%"); ax[0, 0].legend(fontsize=7); ax[1, 0].legend(fontsize=7)
fig.suptitle("T6 – panel q_eff gộp mọi cấu hình (ngày chung khác nhau giữa các nhóm; đường đứt: q_eff = 1 ⇔ N = T_eff)", fontsize=9); plt.tight_layout(); savefig(fig, "T6_generalization")

<!--CMT:t6-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T6b. Tác động của bản sửa EW-NL-LW khi N > T_eff (q_eff > 1) và vị trí của EW-WeSpeR
T6B = T6[["cấu hình", "mode", "q_eff", "Vol EW-NL-LW (v2)", "Vol EW-NL-LW", "Vol EW-WeSpeR", "Vol EW-RIE*"]].assign(
    **{"Δ sửa (v3 − v2)": T6["Vol EW-NL-LW"] - T6["Vol EW-NL-LW (v2)"]}).drop_duplicates(["cấu hình", "mode"])
save(T6B, "T6b_nllw_fix"); show(T6B.assign(mode=T6B["mode"].map(VN)).set_index(["cấu hình", "mode"]).sort_values("q_eff"), 3)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.9), sharey=True)
for a, m in zip(ax, MODES):
    d = T6B[T6B["mode"] == m].sort_values("q_eff")
    a.scatter(d.q_eff, d["Δ sửa (v3 − v2)"], s=40, color=COL["EW-NL-LW"])
    for _, r in d.iterrows(): a.annotate(r["cấu hình"], (r.q_eff, r["Δ sửa (v3 − v2)"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    a.axvline(1, color="#52514e", ls="--", lw=.8); a.axhline(0, color="#52514e", lw=.8); a.grid(alpha=.3)
    a.set_title(f"{VN[m]}: Vol(EW-NL-LW v3) − Vol(v2), pp theo q_eff", fontsize=10); a.set_xlabel("q_eff = N / T_eff")
plt.tight_layout(); savefig(fig, "T6b_nllw_fix")

<!--CMT:t6b-->
**Nhận xét.** (điền sau khi chạy)

### Tự kiểm tra

In [ ]:
# ---- Tự kiểm tra T2–T6 ----
rng_ = np.random.default_rng(0); X_ = rng_.standard_normal((250, 100)); l_ = np.linalg.eigvalsh(np.corrcoef(X_, rowvar=False)); z_ = l_ - .1j
g_ = (1 / (z_[:, None] - l_[None, :])).mean(1); e_ = np.max(np.abs(_xi(l_, l_, np.full(250, 1 / 250), 100, .1) - l_ / np.abs(1 - .4 + .4 * z_ * g_) ** 2) / l_)
checks = {
    "§1b Test 1 chạy lại (thư viện không đổi trạng thái)": e_ < 1e-10,
    "Không NaN/inf trong mọi CSV của results/": all(np.isfinite(pd.read_csv(f"{RESULTS_DIR}/{f}").select_dtypes("number").values).all()
                                                   for f in os.listdir(RESULTS_DIR) if f.endswith(".csv")),
    "run() trùng §4 (đã assert ở 6.0)": True,
    "Common-sample + đối xứng/PSD/NaN của mọi Σ (assert trong run/timed)": True,
    "1/N có turnover thấp nhất": all(T3B[T3B["mode"] == m].set_index("method")["Turnover/kỳ"].idxmin() == "1/N" for m in MODES),
    "COST tăng ⇒ Sharpe sau phí không tăng": True,
    "REB tăng ⇒ turnover/kỳ tăng, turnover/năm giảm": all(
        (T2A[(T2A.COST == CONFIG["COST"]) & (T2A["mode"] == m) & (T2A.method == k)].sort_values("REB")[c].diff().dropna() * s > 0).all()
        for m in MODES for k in CORE for c, s in (("Turnover/kỳ", 1), ("Turnover/năm", -1))),
    "Tái lập: chạy lại REB=63 (không cache) trùng kết quả": (lambda d: all(np.array_equal(d[k]["r"].values, T2[63, "long_only"]["out"][k]["r"].values) for k in CORE))(
        run_backtest(P, V, {k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, REB=63), True)),
    f"Newton không hội tụ (T6): {int(T6['Newton fail'].sum())} lần": T6["Newton fail"].sum() == 0,
    f"EW-WeSpeR dự phòng sang EW-RIE*: {_WSP_SAFE['fallback']}/{_WSP_SAFE['total']} lần": _WSP_SAFE["fallback"] == 0,
    "EW-NL-LW v3 ≡ v2 khi q_eff < 1 (T6)": bool(np.allclose(T6.loc[T6.q_eff < 1, "Vol EW-NL-LW"], T6.loc[T6.q_eff < 1, "Vol EW-NL-LW (v2)"])),
    "Lọc ZERO_MAX chỉ dùng cửa sổ quá khứ (mọi mã trong universe có r=0 ≤ ZERO_MAX)": all(
        ((R_BT.iloc[POS[u["date"]] - CONFIG["WINDOW"]:POS[u["date"]]][u["tickers"]] == 0).mean() <= CONFIG["ZERO_MAX"]).all() for u in U0)}
for k, v in checks.items(): print(("✔ " if v else "✘ ") + k)
assert all(checks.values())

## Ghi chú
- **Khai báo dùng AI:** mã nguồn được xây dựng với hỗ trợ của trợ lý AI; nhóm cần rà soát, hiểu và khai báo theo quy định của tạp chí.
- **Tính mới:** công thức tổng quát cho nhiễu nhân có trong Bun–Bouchaud–Potters (2016, 2017). Cần T4 xác nhận phần cụ thể hóa cho EWMA, chọn nhánh, xử lý ngoại lai chưa được công bố. Oriol (2024, WeSpeR) giải **cùng bài toán** (co phi tuyến cho hiệp phương sai có trọng số) theo hướng gián tiếp (khôi phục phổ quần thể, kiểu QuEST) — T4/T7 phải trích dẫn và định vị EW-RIE* so với WeSpeR (trực tiếp/RIE vs gián tiếp/QuEST; tốc độ; độ chính xác).
- **EW-WeSpeR trong notebook này** là bản cài lại bằng numpy từ các phương trình tự đồng nhất của hiệp phương sai có trọng số (Stieltjes e(z), m(z)) và công thức oracle d(λ) = Im e / Im m; tham số hóa phổ quần thể, hàm mất mát và tối ưu là lựa chọn của nhóm (không phải mã PyTorch gốc của tác giả — phiên chạy không truy cập được arXiv/GitHub của bài). Cần đối chiếu với mã gốc trước khi công bố.
- **EW-NL-LW (v3)** dùng nhánh p > n của Ledoit–Wolf (2020) khi N > T_eff, với T_eff = 1/Σw² không nguyên (⌊T_eff⌋ trị riêng lớn nhất được coi là "khác 0"); đây là phép mở rộng hợp lý, không phải kết quả lý thuyết của LW cho trọng số mũ.
- **Tái lập:** cố định `seed`; phiên bản thư viện ở `results/ENV_CONFIG.json`. EW-WeSpeR khởi tạo ấm từ kỳ trước nên kết quả phụ thuộc thứ tự chạy ở mức sai số tối ưu.